In [ ]:
import earthaccess
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely import wkt
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from rasterio.plot import show

### PIXC

In [ ]:
import earthaccess
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import re

def filter_best_granules(results):
    """Keep best version: PGD0 > PID0, highest ProductCounter"""
    granule_dict = {}
    
    for r in results:
        url = r.data_links()[0]
        filename = url.split('/')[-1]
        
        # Parse: SWOT_L2_HR_PIXC_040_315_228R_20251023T044410_20251023T044421_PID0_03.nc
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', filename)
        if not match:
            continue
        
        base = match.group(1)
        processor = match.group(2)  # PID0 or PGD0
        counter = int(match.group(3))
        
        if base not in granule_dict:
            granule_dict[base] = (r, processor, counter)
        else:
            _, old_proc, old_cnt = granule_dict[base]
            
            # PGD0 > PID0
            proc_better = (processor == 'PGD0' and old_proc == 'PID0')
            proc_same = (processor == old_proc)
            cnt_better = (counter > old_cnt)
            
            if proc_better or (proc_same and cnt_better):
                granule_dict[base] = (r, processor, counter)
    
    return [r for r, _, _ in granule_dict.values()]


selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
data_dir = Path('/work/pi_kandread_umass_edu/swot-urban/1_data_processing')
parquet_files = sorted(data_dir.glob('fctimp_mean_*_sample.parquet'))
print(f"Found {len(parquet_files)} parquet files\n")

earthaccess.login()

for pq_file in parquet_files:
    fctimp_str = pq_file.stem.split('_')[2]
    fctimp_val = float(fctimp_str)
    
    mask_frac = np.isclose(selected_12["fctimp_mea"], fctimp_val, atol=0.01)
    sample = selected_12.loc[mask_frac].copy()
    
    if sample.empty:
        print(f"fctimp {fctimp_str}: No matching reach, skipping")
        continue
    
    bounds = sample.total_bounds
    bbox = (bounds[0] - 0.1, bounds[1] - 0.1, bounds[2] + 0.1, bounds[3] + 0.1)
    
    df = pd.read_parquet(pq_file)
    dates = set(pd.to_datetime(df['time_str']).dt.date)
    
    print(f"fctimp {fctimp_str}: {len(dates)} dates")
    
    all_results = []
    for date in dates:
        date_str = date.strftime('%Y-%m-%d')
        results = earthaccess.search_data(
            short_name='SWOT_L2_HR_PIXC_D',
            bounding_box=bbox,
            temporal=(date_str, date_str)
        )
        all_results.extend(results)
    
    # Filter to best versions
    filtered = filter_best_granules(all_results)
    
    outfile = f'PIXC_url_v2/swot_PIXC_vD_fctimp_{fctimp_str}_urls.txt'
    output_dir = f'PIXC_url_v2'
    Path(output_dir).mkdir(exist_ok=True)
    with open(outfile, 'w') as f:
        for r in filtered:
            f.write(r.data_links()[0] + '\n')
    
    print(f"  → {len(all_results)} → {len(filtered)} granules → {outfile}\n")

### PIXCVec

In [ ]:
import earthaccess
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import re

def filter_best_granules(results):
    """Keep best version: PGD0 > PID0, highest ProductCounter"""
    granule_dict = {}
    
    for r in results:
        url = r.data_links()[0]
        filename = url.split('/')[-1]
        
        # Parse: SWOT_L2_HR_PIXC_040_315_228R_20251023T044410_20251023T044421_PID0_03.nc
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', filename)
        if not match:
            continue
        
        base = match.group(1)
        processor = match.group(2)  # PID0 or PGD0
        counter = int(match.group(3))
        
        if base not in granule_dict:
            granule_dict[base] = (r, processor, counter)
        else:
            _, old_proc, old_cnt = granule_dict[base]
            
            # PGD0 > PID0
            proc_better = (processor == 'PGD0' and old_proc == 'PID0')
            proc_same = (processor == old_proc)
            cnt_better = (counter > old_cnt)
            
            if proc_better or (proc_same and cnt_better):
                granule_dict[base] = (r, processor, counter)
    
    return [r for r, _, _ in granule_dict.values()]


selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
data_dir = Path('/work/pi_kandread_umass_edu/swot-urban/1_data_processing')
parquet_files = sorted(data_dir.glob('fctimp_mean_*_sample.parquet'))
print(f"Found {len(parquet_files)} parquet files\n")

earthaccess.login()

for pq_file in parquet_files:
    fctimp_str = pq_file.stem.split('_')[2]
    fctimp_val = float(fctimp_str)
    
    mask_frac = np.isclose(selected_12["fctimp_mea"], fctimp_val, atol=0.01)
    sample = selected_12.loc[mask_frac].copy()
    
    if sample.empty:
        print(f"fctimp {fctimp_str}: No matching reach, skipping")
        continue
    
    bounds = sample.total_bounds
    bbox = (bounds[0] - 0.1, bounds[1] - 0.1, bounds[2] + 0.1, bounds[3] + 0.1)
    
    df = pd.read_parquet(pq_file)
    dates = set(pd.to_datetime(df['time_str']).dt.date)
    
    print(f"fctimp {fctimp_str}: {len(dates)} dates")
    
    all_results = []
    for date in dates:
        date_str = date.strftime('%Y-%m-%d')
        results = earthaccess.search_data(
            short_name='SWOT_L2_HR_PIXCVec_D',
            bounding_box=bbox,
            temporal=(date_str, date_str)
        )
        all_results.extend(results)
    
    # Filter to best versions
    filtered = filter_best_granules(all_results)
    outfile = f'PIXCVec_url_v2/swot_PIXCVec_vD_fctimp_{fctimp_str}_urls.txt'
    output_dir = f'PIXCVec_url_v2'
    Path(output_dir).mkdir(exist_ok=True)
    with open(outfile, 'w') as f:
        for r in filtered:
            f.write(r.data_links()[0] + '\n')
    
    print(f"  → {len(all_results)} → {len(filtered)} granules → {outfile}\n")

### Height Comparison by legneds

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
from shapely.geometry import Point
import re

def filter_best_pixc_files(files):
    file_dict = {}
    for f in files:
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', f.name)
        if not match:
            continue
        base, processor, counter = match.group(1), match.group(2), int(match.group(3))
        if base not in file_dict:
            file_dict[base] = (f, processor, counter)
        else:
            _, old_proc, old_cnt = file_dict[base]
            if (processor == 'PGD0' and old_proc == 'PID0') or (processor == old_proc and counter > old_cnt):
                file_dict[base] = (f, processor, counter)
    return [f for f, _, _ in file_dict.values()]

def get_tile_from_filename(filename):
    parts = filename.split('_')
    if len(parts) >= 7:
        return parts[6]
    return 'UNK'

def match_and_group_pixc(time_str, files, crid, window_minutes=5):
    dt = pd.to_datetime(time_str)
    if dt.tzinfo:
        dt = dt.tz_localize(None)
    start, end = dt - timedelta(minutes=window_minutes), dt + timedelta(minutes=window_minutes)
    groups = {}
    for f in files:
        parts = f.name.split('_')
        if len(parts) >= 10:
            try:
                fs, fe = pd.to_datetime(parts[7], format='%Y%m%dT%H%M%S'), pd.to_datetime(parts[8], format='%Y%m%dT%H%M%S')
                if not (fe < start or fs > end) and parts[9] == crid:
                    key = f"{parts[7]}_{parts[8]}"
                    groups.setdefault(key, []).append(f)
            except:
                continue
    return groups

def find_matching_pixcvec(pixc_file, pixcvec_files):
    pixc_name = pixc_file.name.replace('PIXC_', 'PIXCVec_')
    for pvf in pixcvec_files:
        if pvf.name == pixc_name:
            return pvf
    return None

def extract_selected_pixels(pixc_file, pixcvec_file, polygon_gdf):
    """Extract ONLY selected pixels (via PIXCVec) within polygon"""
    ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    lon, lat = ds['longitude'].values, ds['latitude'].values
    bounds = polygon_gdf.total_bounds
    
    bbox_mask = ((lon >= bounds[0]) & (lon <= bounds[2]) & (lat >= bounds[1]) & (lat <= bounds[3]))
    if bbox_mask.sum() == 0:
        ds.close()
        return None
    
    classification = ds['classification'].values[bbox_mask]
    water_mask = classification > 2
    if water_mask.sum() == 0:
        ds.close()
        return None
    
    lon_filt, lat_filt = lon[bbox_mask][water_mask], lat[bbox_mask][water_mask]
    poly = polygon_gdf.geometry.iloc[0]
    inside_mask = np.array([poly.contains(Point(x, y)) for x, y in zip(lon_filt, lat_filt)])
    if inside_mask.sum() == 0:
        ds.close()
        return None
    
    final_idx = np.where(bbox_mask)[0][water_mask][inside_mask]
    
    # Get PIXCVec selection
    ds_vec = xr.open_dataset(pixcvec_file)
    reach_id = ds_vec['reach_id'].values[final_idx]
    node_id = ds_vec['node_id'].values[final_idx]
    obs_id = ds_vec['obs_id'].values[final_idx]
    lake_id = ds_vec['lake_id'].values[final_idx]
    
    def is_valid(arr):
        return np.array([len(x) > 0 and x != b'' for x in arr]) if arr.dtype.kind == 'S' else arr > 0
    
    selected_mask = is_valid(reach_id) & is_valid(node_id) & ~is_valid(obs_id) & ~is_valid(lake_id)
    
    # Apply selection mask - ONLY return selected pixels
    selected_idx = final_idx[selected_mask]
    
    if len(selected_idx) == 0:
        ds.close()
        ds_vec.close()
        return None
    
    result = {
        'lon': lon[selected_idx],
        'lat': lat[selected_idx],
        'height': ds['height'].values[selected_idx],
        'classification': ds['classification'].values[selected_idx],
    }
    
    ds.close()
    ds_vec.close()
    return result

def extract_and_merge_selected_pixels(pixc_files, pixcvec_files, polygon_gdf):
    """Extract and merge selected pixels from multiple tiles"""
    all_data = []
    for pf, pvf in zip(pixc_files, pixcvec_files):
        data = extract_selected_pixels(pf, pvf, polygon_gdf)
        if data:
            all_data.append(data)
    
    if not all_data:
        return None
    if len(all_data) == 1:
        return all_data[0]
    
    merged = {k: np.concatenate([d[k] for d in all_data]) for k in all_data[0].keys()}
    return merged

def plot_height_by_classification(normal_data, spike_data, normal_info, spike_info, output_dir, time_str, fctimp_str):
    """
    Boxplot of height distribution by classification for Normal vs Spike
    Only for selected pixels with classification 3, 4, 6, 7
    """
    
    # Classification mapping
    class_map = {
        3: 'water_near_land',
        4: 'open_water',
        6: 'low_coh_water_near_land',
        7: 'open_low_coh_water'
    }
    class_ids = [3, 4, 6, 7]
    class_labels = [class_map[c] for c in class_ids]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for ax, (data, info, label) in zip(axes, [
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        color = 'green' if label == 'NORMAL' else 'red'
        
        # Filter valid heights
        valid_mask = np.isfinite(data['height'])
        heights = data['height'][valid_mask]
        classifications = data['classification'][valid_mask]
        
        # Get heights by classification
        height_by_class = []
        sample_counts = []
        for c in class_ids:
            h = heights[classifications == c]
            height_by_class.append(h)
            sample_counts.append(len(h))
        
        # Total statistics
        total_count = len(heights)
        mean_height = np.mean(heights) if total_count > 0 else np.nan
        
        # Create boxplot
        bp = ax.boxplot(height_by_class, positions=range(len(class_ids)), 
                        widths=0.6, patch_artist=True)
        
        # Style boxplot
        for box in bp['boxes']:
            box.set_facecolor(color)
            box.set_alpha(0.5)
        for whisker in bp['whiskers']:
            whisker.set_color(color)
        for cap in bp['caps']:
            cap.set_color(color)
        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(2)
        for flier in bp['fliers']:
            flier.set_markerfacecolor(color)
            flier.set_markeredgecolor(color)
            flier.set_alpha(0.3)
        
        # Add sample count above each box
        for i, (pos, count) in enumerate(zip(range(len(class_ids)), sample_counts)):
            # Get the upper whisker position for label placement
            if count > 0:
                upper = np.percentile(height_by_class[i], 75) + 1.5 * (np.percentile(height_by_class[i], 75) - np.percentile(height_by_class[i], 25))
                upper = min(upper, np.max(height_by_class[i]))
            else:
                upper = 0
            ax.text(pos, upper + 0.5, f'n={count}', ha='center', va='bottom', 
                    fontsize=9, fontweight='bold', color='black')
        
        # Add mean height line
        if np.isfinite(mean_height):
            ax.axhline(y=mean_height, color='red', linestyle='--', linewidth=2, alpha=0.7, zorder=5)
        
        # Add total statistics text (top right)
        stats_text = f'Total: {total_count} pixels\nMean: {mean_height:.2f} m'
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, 
                ha='right', va='top', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='red', alpha=0.9))
        
        # Labels and title
        ax.set_xticks(range(len(class_ids)))
        ax.set_xticklabels(class_labels, rotation=30, ha='right', fontsize=9)
        ax.set_ylabel('Height (m)', fontsize=11)
        ax.set_xlabel('Classification', fontsize=11)
        ax.set_title(f'{label} | {info["heading"]:.0f}°-{info["tile"]}\nError: {info["error"]:.2f}m', 
                     fontsize=12, fontweight='bold', color=color)
        ax.grid(axis='y', alpha=0.3)
        
        # Set y-axis limits with some padding
        all_heights = np.concatenate([h for h in height_by_class if len(h) > 0])
        if len(all_heights) > 0:
            ymin = np.percentile(all_heights, 1) - 1
            ymax = np.percentile(all_heights, 99) + 2
            ax.set_ylim(ymin, ymax)
    
    fig.suptitle(f'ISA={fctimp_str} | Height Distribution by Classification (Selected Pixels Only)\n'
                 f'Normal: {normal_info["time_str"]} vs Spike: {spike_info["time_str"]}', 
                 fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(f'{output_dir}/COMPARE_{time_str}_HeightByClass.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  Saved: {output_dir}/COMPARE_{time_str}_HeightByClass.png")

def plot_height_cdf_by_classification(normal_data, spike_data, normal_info, spike_info,
                                      output_dir, time_str, fctimp_str):
    """
    Height ECDF by classification for Normal vs Spike
    - 4개의 세로 subplot (water_near_land, open_water, low_coh_water_near_land, open_low_coh_water)
    - x축(Height[m])은 공유
    - 각 subplot 안에 Normal/Spike CDF + 각 그룹 mean 세로선
    - 추가: 전체 Normal mean, 전체 Spike mean 세로선(모든 subplot 공통)
    """

    # Classification mapping
    class_map = {
        3: 'water_near_land',
        4: 'open_water',
        6: 'low_coh_water_near_land',
        7: 'open_low_coh_water'
    }
    class_ids = [3, 4, 6, 7]

    # helper: 클래스별 height 분리
    def split_by_class(data):
        mask = np.isfinite(data['height'])
        h = data['height'][mask]
        cls = data['classification'][mask]
        out = {}
        for cid in class_ids:
            out[cid] = h[cls == cid]
        return out

    normal_by_class = split_by_class(normal_data)
    spike_by_class  = split_by_class(spike_data)

    # ---- 전체 Normal / Spike 높이 모아서 global mean 계산 ----
    norm_all_list = [v for v in normal_by_class.values() if len(v) > 0]
    spike_all_list = [v for v in spike_by_class.values() if len(v) > 0]

    global_mean_normal = np.mean(np.concatenate(norm_all_list)) if len(norm_all_list) > 0 else None
    global_mean_spike  = np.mean(np.concatenate(spike_all_list)) if len(spike_all_list) > 0 else None

    # ---- x축 범위 (모든 높이 합쳐서) ----
    all_heights = []
    for d in (normal_by_class, spike_by_class):
        for arr in d.values():
            if len(arr) > 0:
                all_heights.append(arr)
    if len(all_heights) == 0:
        print("  No valid heights to plot CDF.")
        return
    all_heights = np.concatenate(all_heights)
    xmin = np.percentile(all_heights, 1) - 0.5
    xmax = np.percentile(all_heights, 99) + 0.5

    # figure: 세로 4개, x축 공유
    fig, axes = plt.subplots(len(class_ids), 1, figsize=(14, 10), sharex=True)
    if len(class_ids) == 1:
        axes = [axes]

    for ax, cid in zip(axes, class_ids):
        label = class_map[cid]

        h_norm = normal_by_class[cid]
        h_spike = spike_by_class[cid]

        # CDF 함수
        def ecdf(a):
            a_sort = np.sort(a)
            y = np.arange(1, len(a_sort) + 1) / len(a_sort)
            return a_sort, y

        plotted_any = False
        handles = []
        labels = []

        # Normal CDF
        if len(h_norm) > 0:
            x_n, y_n = ecdf(h_norm)
            ln, = ax.step(x_n, y_n, where='post', color='green',
                          label=f'Normal (n={len(h_norm)})')
            mean_n = np.mean(h_norm)
            # per-class mean
            ax.axvline(mean_n, color='green', linestyle='--', alpha=0.7)
            handles.append(ln)
            labels.append(f'Normal (n={len(h_norm)}, μ={mean_n:.2f} m)')
            plotted_any = True

        # Spike CDF
        if len(h_spike) > 0:
            x_s, y_s = ecdf(h_spike)
            ls, = ax.step(x_s, y_s, where='post', color='red',
                          label=f'Spike (n={len(h_spike)})')
            mean_s = np.mean(h_spike)
            # per-class mean
            ax.axvline(mean_s, color='red', linestyle='--', alpha=0.7)
            handles.append(ls)
            labels.append(f'Spike (n={len(h_spike)}, μ={mean_s:.2f} m)')
            plotted_any = True

        # ---- 전체 Normal/Spike 평균 세로선 (모든 subplot 공통) ----
        # 첫 번째 subplot에만 legend entry 추가
        is_first_axis = (cid == class_ids[0])

        if global_mean_normal is not None:
            gln = ax.axvline(global_mean_normal,
                             color='green', linestyle=':', linewidth=1.5, alpha=0.6)
            if is_first_axis:
                handles.append(gln)
                labels.append(f'Global Normal μ={global_mean_normal:.2f} m')

        if global_mean_spike is not None:
            gls = ax.axvline(global_mean_spike,
                             color='red', linestyle=':', linewidth=1.5, alpha=0.6)
            if is_first_axis:
                handles.append(gls)
                labels.append(f'Global Spike μ={global_mean_spike:.2f} m')

        if not plotted_any:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11)
        else:
            ax.legend(handles, labels, loc='lower right', fontsize=9)

        ax.set_ylabel('CDF', fontsize=11)
        ax.set_title(f'{label}', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Height (m)', fontsize=12)
    axes[-1].set_xlim(xmin, xmax)

    # 전체 제목 (Normal/Spike 정보 포함)
    fig.suptitle(
        f'ISA={fctimp_str} | Height CDF by Classification (Selected Pixels Only)\n'
        f'Normal: {normal_info["time_str"]} | Heading {normal_info["heading"]:.0f}°-{normal_info["tile"]} | Error={normal_info["error"]:.2f} m\n'
        f'Spike:  {spike_info["time_str"]} | Heading {spike_info["heading"]:.0f}°-{spike_info["tile"]} | Error={spike_info["error"]:.2f} m',
        fontsize=12, fontweight='bold'
    )

    plt.tight_layout(rect=[0, 0, 1, 0.88])

    out_path = f'{output_dir}/COMPARE_{time_str}_HeightCDFByClass.png'
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"  Saved: {out_path}")



def process_fctimp_height_analysis(fctimp_str, spike_threshold=1.0, output_dir=None):
    """Main function to process height by classification analysis"""
    
    if output_dir is None:
        output_dir = f'isa_{fctimp_str}_v3'
    Path(output_dir).mkdir(exist_ok=True)
    
    # Load reach polygon
    selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
    fctimp_val = float(fctimp_str)
    sample = selected_12[np.isclose(selected_12["fctimp_mea"], fctimp_val, atol=0.01)].copy()
    if sample.empty:
        print(f"No reach for {fctimp_str}")
        return
    
    # Load time series data
    df = pd.read_parquet(f'/work/pi_kandread_umass_edu/swot-urban/1_data_processing/fctimp_mean_{fctimp_str}_sample.parquet')
    d = df.copy()
    d['time_dt'] = pd.to_datetime(d['time_str'])
    d = d.sort_values('time_dt').reset_index(drop=True)
    d['error'] = d['wse_relative_m'] - d['wse_usgs_relative_m']  # + = overestimate, - = underestimate
    d['is_spike'] = d['error'] >= spike_threshold
    
    # Load PIXC files
    pixc_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXC/fctimp_{fctimp_str}')
    pixcvec_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXCVec/fctimp_{fctimp_str}')
    pixc_files = filter_best_pixc_files(list(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc')))
    pixcvec_files = filter_best_pixc_files(list(pixcvec_dir.glob('SWOT_L2_HR_PIXCVec_*.nc')))
    
    polygon = sample.to_crs('EPSG:4326')
    
    print(f"\n{'='*70}")
    print(f"Processing ISA={fctimp_str}")
    print(f"Spikes: {d['is_spike'].sum()}, Normals: {(~d['is_spike']).sum()}")
    print(f"{'='*70}")
    
    for idx, spike_row in d[d['is_spike']].iterrows():
        print(f"\nSpike: {spike_row['time_str']} (error={spike_row['error']:.2f}m)")
        
        # Find closest normal
        normals = d[~d['is_spike']].copy()
        if len(normals) == 0:
            continue
        normals['time_diff'] = (normals['time_dt'] - spike_row['time_dt']).abs()
        closest_normal = normals.loc[normals['time_diff'].idxmin()]
        print(f"Normal: {closest_normal['time_str']} (error={closest_normal['error']:.2f}m)")
        
        # Get Normal data (selected pixels only)
        normal_data = None
        normal_info = None
        for group_key, nf_list in match_and_group_pixc(closest_normal['time_str'], pixc_files, closest_normal['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(nf, pixcvec_files) for nf in nf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_selected_pixels(nf_list, vec_list, polygon)
            if data:
                # Get heading from first file
                ds_tvp = xr.open_dataset(nf_list[0], group='tvp')
                heading = np.nanmedian(ds_tvp['velocity_heading'].values)
                ds_tvp.close()
                
                normal_data = data
                normal_info = {
                    'time_str': closest_normal['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': closest_normal['time_dt'],
                    'crid': closest_normal['crid'],
                    'error': closest_normal['error'],
                    'tile': get_tile_from_filename(nf_list[0].name),
                    'heading': heading,
                }
                break
        
        # Get Spike data (selected pixels only)
        spike_data = None
        spike_info = None
        for group_key, sf_list in match_and_group_pixc(spike_row['time_str'], pixc_files, spike_row['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(sf, pixcvec_files) for sf in sf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_selected_pixels(sf_list, vec_list, polygon)
            if data:
                # Get heading from first file
                ds_tvp = xr.open_dataset(sf_list[0], group='tvp')
                heading = np.nanmedian(ds_tvp['velocity_heading'].values)
                ds_tvp.close()
                
                spike_data = data
                spike_info = {
                    'time_str': spike_row['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': spike_row['time_dt'],
                    'crid': spike_row['crid'],
                    'error': spike_row['error'],
                    'tile': get_tile_from_filename(sf_list[0].name),
                    'heading': heading,
                }
                break
        
        # Generate CDF plot instead of boxplot
        if normal_data and spike_data:
            plot_height_cdf_by_classification(
                normal_data, spike_data,
                normal_info, spike_info,
                output_dir,
                spike_info['time_str'],
                fctimp_str
            )
        else:
            print(f"  Skipped: Missing data (Normal={normal_data is not None}, Spike={spike_data is not None})")

    print(f"\n✓ Done: {output_dir}")


# Run for ISA=0.01
if __name__ == "__main__":
    process_fctimp_height_analysis('0.01', spike_threshold=0.1832, output_dir='isa_0.01_v3')

In [ ]:
for fc in ['0.01', '0.06', '0.12', '0.17', '0.22', '0.32', '0.28', '0.40', '0.45', '0.50', '0.55', '0.61']:
    process_fctimp_height_analysis(fc, spike_threshold=0.1832, output_dir=f'isa_{fc}_v3')

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
from shapely.geometry import Point
from scipy.spatial import cKDTree
import re

def filter_best_pixc_files(files):
    file_dict = {}
    for f in files:
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', f.name)
        if not match:
            continue
        base, processor, counter = match.group(1), match.group(2), int(match.group(3))
        if base not in file_dict:
            file_dict[base] = (f, processor, counter)
        else:
            _, old_proc, old_cnt = file_dict[base]
            if (processor == 'PGD0' and old_proc == 'PID0') or (processor == old_proc and counter > old_cnt):
                file_dict[base] = (f, processor, counter)
    return [f for f, _, _ in file_dict.values()]

def get_tile_from_filename(filename):
    parts = filename.split('_')
    if len(parts) >= 7:
        return parts[6]
    return 'UNK'

def match_and_group_pixc(time_str, files, crid, window_minutes=5):
    dt = pd.to_datetime(time_str)
    if dt.tzinfo:
        dt = dt.tz_localize(None)
    start, end = dt - timedelta(minutes=window_minutes), dt + timedelta(minutes=window_minutes)
    groups = {}
    for f in files:
        parts = f.name.split('_')
        if len(parts) >= 10:
            try:
                fs, fe = pd.to_datetime(parts[7], format='%Y%m%dT%H%M%S'), pd.to_datetime(parts[8], format='%Y%m%dT%H%M%S')
                if not (fe < start or fs > end) and parts[9] == crid:
                    key = f"{parts[7]}_{parts[8]}"
                    groups.setdefault(key, []).append(f)
            except:
                continue
    return groups

def find_matching_pixcvec(pixc_file, pixcvec_files):
    pixc_name = pixc_file.name.replace('PIXC_', 'PIXCVec_')
    for pvf in pixcvec_files:
        if pvf.name == pixc_name:
            return pvf
    return None

def extract_selected_pixels(pixc_file, pixcvec_file, polygon_gdf):
    """Extract ONLY selected pixels (via PIXCVec) within polygon"""
    ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    lon, lat = ds['longitude'].values, ds['latitude'].values
    bounds = polygon_gdf.total_bounds
    
    bbox_mask = ((lon >= bounds[0]) & (lon <= bounds[2]) & (lat >= bounds[1]) & (lat <= bounds[3]))
    if bbox_mask.sum() == 0:
        ds.close()
        return None
    
    classification = ds['classification'].values[bbox_mask]
    water_mask = classification > 2
    if water_mask.sum() == 0:
        ds.close()
        return None
    
    lon_filt, lat_filt = lon[bbox_mask][water_mask], lat[bbox_mask][water_mask]
    poly = polygon_gdf.geometry.iloc[0]
    inside_mask = np.array([poly.contains(Point(x, y)) for x, y in zip(lon_filt, lat_filt)])
    if inside_mask.sum() == 0:
        ds.close()
        return None
    
    final_idx = np.where(bbox_mask)[0][water_mask][inside_mask]
    
    # Get PIXCVec selection
    ds_vec = xr.open_dataset(pixcvec_file)
    reach_id = ds_vec['reach_id'].values[final_idx]
    node_id = ds_vec['node_id'].values[final_idx]
    obs_id = ds_vec['obs_id'].values[final_idx]
    lake_id = ds_vec['lake_id'].values[final_idx]
    
    def is_valid(arr):
        return np.array([len(x) > 0 and x != b'' for x in arr]) if arr.dtype.kind == 'S' else arr > 0
    
    selected_mask = is_valid(reach_id) & is_valid(node_id) & ~is_valid(obs_id) & ~is_valid(lake_id)
    
    # Apply selection mask - ONLY return selected pixels
    selected_idx = final_idx[selected_mask]
    
    if len(selected_idx) == 0:
        ds.close()
        ds_vec.close()
        return None
    
    result = {
        'lon': lon[selected_idx],
        'lat': lat[selected_idx],
        'height': ds['height'].values[selected_idx],
        'classification': ds['classification'].values[selected_idx],
    }
    
    ds.close()
    ds_vec.close()
    return result

def extract_and_merge_selected_pixels(pixc_files, pixcvec_files, polygon_gdf):
    """Extract and merge selected pixels from multiple tiles"""
    all_data = []
    for pf, pvf in zip(pixc_files, pixcvec_files):
        data = extract_selected_pixels(pf, pvf, polygon_gdf)
        if data:
            all_data.append(data)
    
    if not all_data:
        return None
    if len(all_data) == 1:
        return all_data[0]
    
    merged = {k: np.concatenate([d[k] for d in all_data]) for k in all_data[0].keys()}
    return merged


def match_pixels_nearest_neighbor(normal_data, spike_data, threshold_deg=0.0001):
    """
    Match pixels between Normal and Spike using Nearest Neighbor (KD-Tree)
    
    Parameters:
        normal_data: dict with 'lon', 'lat' arrays
        spike_data: dict with 'lon', 'lat' arrays
        threshold_deg: max distance in degrees (~11m at 0.0001°)
    
    Returns:
        normal_idx: indices of matched Normal pixels
        spike_idx: indices of matched Spike pixels
        distances: distances between matched pairs
    """
    # Build KD-Tree from Normal pixels
    normal_coords = np.column_stack([normal_data['lon'], normal_data['lat']])
    spike_coords = np.column_stack([spike_data['lon'], spike_data['lat']])
    
    normal_tree = cKDTree(normal_coords)
    
    # Find nearest Normal pixel for each Spike pixel
    distances, normal_indices = normal_tree.query(spike_coords, k=1)
    
    # Filter by threshold
    matched_mask = distances <= threshold_deg
    
    # Get matched indices
    spike_idx = np.where(matched_mask)[0]
    normal_idx = normal_indices[matched_mask]
    matched_distances = distances[matched_mask]
    
    return normal_idx, spike_idx, matched_distances


def plot_matched_pixel_analysis(normal_data, spike_data, normal_info, spike_info, output_dir, time_str, fctimp_str):
    """
    Analyze matched pixels between Normal and Spike
    1. Find same-location pixels using Nearest Neighbor
    2. Calculate height difference from open_water-to-open_water pixels
    3. Compare classification changes with normalized heights
    """
    
    # Step 1: Match pixels using Nearest Neighbor
    # threshold_deg=0.0001 ≈ 11m at equator
    threshold_deg = 0.0001
    normal_idx, spike_idx, distances = match_pixels_nearest_neighbor(
        normal_data, spike_data, threshold_deg=threshold_deg
    )
    
    if len(normal_idx) == 0:
        print(f"  No matched pixels found within threshold ({threshold_deg}° ≈ {threshold_deg * 111000:.1f}m)!")
        return
    
    mean_dist_m = np.mean(distances) * 111000  # approximate meters
    max_dist_m = np.max(distances) * 111000
    
    print(f"  Matched {len(normal_idx)} pixels (threshold: {threshold_deg}° ≈ {threshold_deg * 111000:.1f}m)")
    print(f"  Mean distance: {mean_dist_m:.2f}m, Max distance: {max_dist_m:.2f}m")
    
    # Extract matched data
    matched_normal = {
        'lon': normal_data['lon'][normal_idx],
        'lat': normal_data['lat'][normal_idx],
        'height': normal_data['height'][normal_idx],
        'classification': normal_data['classification'][normal_idx],
    }
    matched_spike = {
        'lon': spike_data['lon'][spike_idx],
        'lat': spike_data['lat'][spike_idx],
        'height': spike_data['height'][spike_idx],
        'classification': spike_data['classification'][spike_idx],
    }
    
    # Step 2: Find open_water (class 4) in BOTH Normal and Spike
    # These are "stable" pixels for calculating baseline height difference
    open_water_both = (matched_normal['classification'] == 4) & (matched_spike['classification'] == 4)
    n_open_water_both = open_water_both.sum()
    
    if n_open_water_both == 0:
        print(f"  No open_water-to-open_water pixels found!")
        return
    
    # Calculate mean height difference (Spike - Normal) for open_water pixels
    height_diff_open_water = matched_spike['height'][open_water_both] - matched_normal['height'][open_water_both]
    mean_height_diff = np.nanmean(height_diff_open_water)
    
    print(f"  Open water (both): {n_open_water_both} pixels")
    print(f"  Mean height difference (Spike - Normal): {mean_height_diff:.3f} m")
    
    # Step 3: Find pixels that are open_water in Normal but changed in Spike
    # Normal: open_water (4) -> Spike: low_coh_water_near_land (6) or open_low_coh_water (7)
    normal_open_spike_lcwnl = (matched_normal['classification'] == 4) & (matched_spike['classification'] == 6)
    normal_open_spike_olcw = (matched_normal['classification'] == 4) & (matched_spike['classification'] == 7)
    
    n_lcwnl = normal_open_spike_lcwnl.sum()
    n_olcw = normal_open_spike_olcw.sum()
    
    print(f"  Normal(open_water) -> Spike(low_coh_water_near_land): {n_lcwnl} pixels")
    print(f"  Normal(open_water) -> Spike(open_low_coh_water): {n_olcw} pixels")
    
    # Get heights for each group
    # Normalize spike heights by subtracting mean_height_diff
    heights_normal_open = matched_normal['height'][open_water_both]
    heights_spike_open = matched_spike['height'][open_water_both] - mean_height_diff  # normalized
    
    heights_normal_to_lcwnl = matched_normal['height'][normal_open_spike_lcwnl]
    heights_spike_lcwnl = matched_spike['height'][normal_open_spike_lcwnl] - mean_height_diff  # normalized
    
    heights_normal_to_olcw = matched_normal['height'][normal_open_spike_olcw]
    heights_spike_olcw = matched_spike['height'][normal_open_spike_olcw] - mean_height_diff  # normalized
    
    # =========================================================================
    # Additional classification groups: water_near_land (class 3)
    # =========================================================================
    # Both water_near_land (class 3)
    water_near_land_both = (matched_normal['classification'] == 3) & (matched_spike['classification'] == 3)
    n_wnl_both = water_near_land_both.sum()
    
    heights_normal_wnl = matched_normal['height'][water_near_land_both]
    heights_spike_wnl_raw = matched_spike['height'][water_near_land_both]  # RAW (no correction)
    
    print(f"  Water near land (both): {n_wnl_both} pixels")
    
    # RAW spike heights for open_water (before correction)
    heights_spike_open_raw = matched_spike['height'][open_water_both]  # RAW
    
    # Calculate overall means (all matched pixels)
    all_normal_heights = matched_normal['height']
    all_spike_heights_raw = matched_spike['height']
    overall_mean_normal = np.nanmean(all_normal_heights)
    overall_mean_spike_raw = np.nanmean(all_spike_heights_raw)
    
    # =========================================================================
    # CDF PLOT: 4 rows with shared x-axis
    # Row 0: both water_near_land (RAW - no correction)
    # Row 1: both open_water (RAW - shows where correction comes from)
    # Row 2: changed to low_coh_water_near_land (CORRECTED)
    # Row 3: changed to open_low_coh_water (CORRECTED)
    # =========================================================================
    
    def plot_cdf(ax, data, color, label, linestyle='-', linewidth=2):
        """Plot CDF on given axis"""
        if len(data) == 0:
            return None, None
        sorted_data = np.sort(data[np.isfinite(data)])
        cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        ax.plot(sorted_data, cdf, color=color, linestyle=linestyle, linewidth=linewidth, label=label)
        return np.nanmean(data), len(data)
    
    # Collect all heights to determine shared x-axis range
    all_heights = []
    if n_wnl_both > 0:
        all_heights.extend([heights_normal_wnl, heights_spike_wnl_raw])
    if n_open_water_both > 0:
        all_heights.extend([heights_normal_open, heights_spike_open_raw])
    if n_lcwnl > 0:
        all_heights.extend([heights_normal_to_lcwnl, heights_spike_lcwnl])
    if n_olcw > 0:
        all_heights.extend([heights_normal_to_olcw, heights_spike_olcw])
    
    if len(all_heights) == 0:
        print(f"  No data for CDF plot!")
        return
    
    all_heights_flat = np.concatenate([h for h in all_heights if len(h) > 0])
    x_min = np.nanpercentile(all_heights_flat, 1) - 0.5
    x_max = np.nanpercentile(all_heights_flat, 99) + 0.5
    
    # Create figure with 4 rows
    fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
    
    row_configs = [
        {
            'title': 'Both water_near_land (class 3) - RAW',
            'normal_data': heights_normal_wnl if n_wnl_both > 0 else np.array([]),
            'spike_data': heights_spike_wnl_raw if n_wnl_both > 0 else np.array([]),
            'n_pixels': n_wnl_both,
            'is_corrected': False,
            'spike_label': 'Spike RAW',
        },
        {
            'title': f'Both open_water (class 4) - RAW → Height correction calculated here: Δ={mean_height_diff:.3f}m',
            'normal_data': heights_normal_open if n_open_water_both > 0 else np.array([]),
            'spike_data': heights_spike_open_raw if n_open_water_both > 0 else np.array([]),
            'n_pixels': n_open_water_both,
            'is_corrected': False,
            'spike_label': 'Spike RAW',
        },
        {
            'title': f'Changed: open_water → low_coh_water_near_land - CORRECTED (Spike - {mean_height_diff:.3f}m)',
            'normal_data': heights_normal_to_lcwnl if n_lcwnl > 0 else np.array([]),
            'spike_data': heights_spike_lcwnl if n_lcwnl > 0 else np.array([]),
            'n_pixels': n_lcwnl,
            'is_corrected': True,
            'spike_label': 'Spike CORRECTED',
        },
        {
            'title': f'Changed: open_water → open_low_coh_water - CORRECTED (Spike - {mean_height_diff:.3f}m)',
            'normal_data': heights_normal_to_olcw if n_olcw > 0 else np.array([]),
            'spike_data': heights_spike_olcw if n_olcw > 0 else np.array([]),
            'n_pixels': n_olcw,
            'is_corrected': True,
            'spike_label': 'Spike CORRECTED',
        },
    ]
    
    for i, (ax, config) in enumerate(zip(axes, row_configs)):
        normal_data = config['normal_data']
        spike_data = config['spike_data']
        n_pixels = config['n_pixels']
        is_corrected = config['is_corrected']
        spike_label = config['spike_label']
        
        if n_pixels > 0 and len(normal_data) > 0 and len(spike_data) > 0:
            # Plot CDFs
            mean_normal, n_normal = plot_cdf(ax, normal_data, 'green', f'Normal (n={len(normal_data)})')
            mean_spike, n_spike = plot_cdf(ax, spike_data, 'red', f'{spike_label} (n={len(spike_data)})')
            
            # Add mean lines as vertical dashed lines
            if mean_normal is not None:
                ax.axvline(x=mean_normal, color='green', linestyle='--', linewidth=1.5, alpha=0.7)
            if mean_spike is not None:
                ax.axvline(x=mean_spike, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
            
            # Add overall mean lines (thinner, different style)
            ax.axvline(x=overall_mean_normal, color='darkgreen', linestyle=':', linewidth=1, alpha=0.5)
            ax.axvline(x=overall_mean_spike_raw, color='darkred', linestyle=':', linewidth=1, alpha=0.5)
            
            # Stats text
            diff = mean_spike - mean_normal if (mean_normal is not None and mean_spike is not None) else 0
            stats_text = f'n={n_pixels} | Normal μ={mean_normal:.2f}m | Spike μ={mean_spike:.2f}m | Δ={diff:.3f}m'
            ax.text(0.98, 0.95, stats_text, transform=ax.transAxes, 
                    ha='right', va='top', fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='gray', alpha=0.9))
            
            # Highlight row 1 (open_water) as the source of correction
            if i == 1:
                ax.set_facecolor('#ffffee')  # Light yellow background
            
            ax.legend(loc='lower right', fontsize=9)
        else:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, 
                    ha='center', va='center', fontsize=12, color='gray')
        
        ax.set_ylabel('CDF', fontsize=10)
        ax.set_title(config['title'], fontsize=10, fontweight='bold', loc='left',
                     color='darkorange' if i == 1 else 'black')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.05)
        ax.set_xlim(x_min, x_max)
    
    # X-axis label only on bottom
    axes[-1].set_xlabel('Height (m)', fontsize=11)
    
    # Main title with overall stats
    fig.suptitle(f'ISA={fctimp_str} | CDF Comparison: Normal vs Spike\n'
                 f'Matched pixels (NN, {threshold_deg}°≈{threshold_deg*111000:.0f}m): {len(normal_idx)} | '
                 f'Overall Mean: Normal={overall_mean_normal:.2f}m, Spike RAW={overall_mean_spike_raw:.2f}m',
                 fontsize=12, fontweight='bold')
    
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(f'{output_dir}/MATCHED_{time_str}_CDF.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  Saved: MATCHED_{time_str}_CDF.png")


def process_fctimp_matched_analysis(fctimp_str, spike_threshold=1.0, output_dir=None):
    """Main function to process matched pixel analysis"""
    
    if output_dir is None:
        output_dir = f'isa_{fctimp_str}_v3'
    Path(output_dir).mkdir(exist_ok=True)
    
    # Load reach polygon
    selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
    fctimp_val = float(fctimp_str)
    sample = selected_12[np.isclose(selected_12["fctimp_mea"], fctimp_val, atol=0.01)].copy()
    if sample.empty:
        print(f"No reach for {fctimp_str}")
        return
    
    # Load time series data
    df = pd.read_parquet(f'/work/pi_kandread_umass_edu/swot-urban/1_data_processing/fctimp_mean_{fctimp_str}_sample.parquet')
    d = df.copy()
    d['time_dt'] = pd.to_datetime(d['time_str'])
    d = d.sort_values('time_dt').reset_index(drop=True)
    d['error'] = d['wse_relative_m'] - d['wse_usgs_relative_m']  # + = overestimate, - = underestimate
    d['is_spike'] = d['error'] >= spike_threshold
    
    # Load PIXC files
    pixc_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXC/fctimp_{fctimp_str}')
    pixcvec_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXCVec/fctimp_{fctimp_str}')
    pixc_files = filter_best_pixc_files(list(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc')))
    pixcvec_files = filter_best_pixc_files(list(pixcvec_dir.glob('SWOT_L2_HR_PIXCVec_*.nc')))
    
    polygon = sample.to_crs('EPSG:4326')
    
    print(f"\n{'='*70}")
    print(f"MATCHED PIXEL ANALYSIS (Nearest Neighbor): ISA={fctimp_str}")
    print(f"Spikes: {d['is_spike'].sum()}, Normals: {(~d['is_spike']).sum()}")
    print(f"{'='*70}")
    
    for idx, spike_row in d[d['is_spike']].iterrows():
        print(f"\n{'-'*50}")
        print(f"Spike: {spike_row['time_str']} (error={spike_row['error']:.2f}m)")
        
        # Find closest normal
        normals = d[~d['is_spike']].copy()
        if len(normals) == 0:
            continue
        normals['time_diff'] = (normals['time_dt'] - spike_row['time_dt']).abs()
        closest_normal = normals.loc[normals['time_diff'].idxmin()]
        print(f"Normal: {closest_normal['time_str']} (error={closest_normal['error']:.2f}m)")
        
        # Get Normal data (selected pixels only)
        normal_data = None
        normal_info = None
        for group_key, nf_list in match_and_group_pixc(closest_normal['time_str'], pixc_files, closest_normal['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(nf, pixcvec_files) for nf in nf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_selected_pixels(nf_list, vec_list, polygon)
            if data:
                ds_tvp = xr.open_dataset(nf_list[0], group='tvp')
                heading = np.nanmedian(ds_tvp['velocity_heading'].values)
                ds_tvp.close()
                
                normal_data = data
                normal_info = {
                    'time_str': closest_normal['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': closest_normal['time_dt'],
                    'error': closest_normal['error'],
                    'tile': get_tile_from_filename(nf_list[0].name),
                    'heading': heading,
                }
                break
        
        # Get Spike data (selected pixels only)
        spike_data = None
        spike_info = None
        for group_key, sf_list in match_and_group_pixc(spike_row['time_str'], pixc_files, spike_row['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(sf, pixcvec_files) for sf in sf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_selected_pixels(sf_list, vec_list, polygon)
            if data:
                ds_tvp = xr.open_dataset(sf_list[0], group='tvp')
                heading = np.nanmedian(ds_tvp['velocity_heading'].values)
                ds_tvp.close()
                
                spike_data = data
                spike_info = {
                    'time_str': spike_row['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': spike_row['time_dt'],
                    'error': spike_row['error'],
                    'tile': get_tile_from_filename(sf_list[0].name),
                    'heading': heading,
                }
                break
        
        # Generate matched pixel analysis
        if normal_data and spike_data:
            print(f"  Normal pixels: {len(normal_data['lon'])}")
            print(f"  Spike pixels: {len(spike_data['lon'])}")
            plot_matched_pixel_analysis(normal_data, spike_data, normal_info, spike_info, 
                                        output_dir, spike_info['time_str'], fctimp_str)
        else:
            print(f"  Skipped: Missing data")
    
    print(f"\n✓ Done: {output_dir}")


# Run for ISA=0.01
if __name__ == "__main__":
    process_fctimp_matched_analysis('0.01', spike_threshold=0.1832, output_dir='isa_0.01_v3')

In [ ]:
for fc in ['0.01', '0.06', '0.12', '0.17', '0.22', '0.32', '0.28', '0.40', '0.45', '0.50', '0.55', '0.61']:
    process_fctimp_matched_analysis(fc, spike_threshold=0.1832, output_dir=f'isa_{fc}_v3')

### PIXC SPIKE & NORMAL BY ISA

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
from shapely.geometry import Point
import rasterio
import re
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import math

def filter_best_pixc_files(files):
    file_dict = {}
    for f in files:
        match = re.match(r'(.+)_(P[IG]D\d)_(\d+)\.nc', f.name)
        if not match:
            continue
        base, processor, counter = match.group(1), match.group(2), int(match.group(3))
        if base not in file_dict:
            file_dict[base] = (f, processor, counter)
        else:
            _, old_proc, old_cnt = file_dict[base]
            if (processor == 'PGD0' and old_proc == 'PID0') or (processor == old_proc and counter > old_cnt):
                file_dict[base] = (f, processor, counter)
    return [f for f, _, _ in file_dict.values()]

def get_tile_from_filename(filename):
    parts = filename.split('_')
    if len(parts) >= 7:
        return parts[6]
    return 'UNK'

def get_heading_from_pixc(pixc_file):
    try:
        ds_tvp = xr.open_dataset(pixc_file, group='tvp')
        heading = np.nanmedian(ds_tvp['velocity_heading'].values)
        ds_tvp.close()
        return heading
    except:
        return None

def calculate_sun_position(dt, lat, lon):
    """
    Calculate sun azimuth and elevation for given datetime and location
    
    Returns:
        azimuth: degrees from North (0=N, 90=E, 180=S, 270=W)
        elevation: degrees above horizon
    """
    # Handle pandas Timestamp
    if hasattr(dt, 'to_pydatetime'):
        dt = dt.to_pydatetime()
    
    # Handle timezone - convert to naive datetime (assuming UTC)
    if hasattr(dt, 'tzinfo') and dt.tzinfo is not None:
        # Remove timezone info (data is already in UTC)
        dt = dt.replace(tzinfo=None)
    
    # Days since J2000.0
    n = (dt - datetime(2000, 1, 1, 12, 0, 0)).total_seconds() / 86400.0
    
    # Mean longitude of the Sun
    L = (280.460 + 0.9856474 * n) % 360
    
    # Mean anomaly of the Sun
    g = math.radians((357.528 + 0.9856003 * n) % 360)
    
    # Ecliptic longitude
    lambda_sun = math.radians(L + 1.915 * math.sin(g) + 0.020 * math.sin(2 * g))
    
    # Obliquity of the ecliptic
    epsilon = math.radians(23.439 - 0.0000004 * n)
    
    # Right ascension and declination
    alpha = math.atan2(math.cos(epsilon) * math.sin(lambda_sun), math.cos(lambda_sun))
    delta = math.asin(math.sin(epsilon) * math.sin(lambda_sun))
    
    # Greenwich Mean Sidereal Time
    GMST = (18.697374558 + 24.06570982441908 * n) % 24
    
    # Local Sidereal Time
    LST = (GMST + lon / 15.0) % 24
    
    # Hour angle
    H = math.radians((LST * 15) - math.degrees(alpha))
    
    # Convert lat to radians
    phi = math.radians(lat)
    
    # Altitude (elevation)
    sin_alt = math.sin(phi) * math.sin(delta) + math.cos(phi) * math.cos(delta) * math.cos(H)
    elevation = math.degrees(math.asin(sin_alt))
    
    # Azimuth
    cos_az = (math.sin(delta) - math.sin(phi) * sin_alt) / (math.cos(phi) * math.cos(math.asin(sin_alt)))
    cos_az = max(-1, min(1, cos_az))  # Clamp to [-1, 1]
    azimuth = math.degrees(math.acos(cos_az))
    
    if math.sin(H) > 0:
        azimuth = 360 - azimuth
    
    return azimuth, elevation

def add_arrows_and_indicators(ax, bounds, heading, sun_azimuth=None, sun_elevation=None, sat_info=None):
    """
    Add North arrow, Satellite position with heading arrow, and Sun position
    - SAT: position based on cross-track sign, arrow shows heading direction
    - SUN: position based on azimuth, arrow points toward plot center
    - N: fixed at top as reference
    """
    # 1. NORTH ARROW - fixed at top right outside
    ax.annotate('', xy=(1.06, 1.08), xytext=(1.06, 0.98),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
                annotation_clip=False, zorder=10)
    ax.text(1.06, 1.10, 'N', transform=ax.transAxes,
            ha='center', va='bottom', fontsize=9, fontweight='bold', color='black', clip_on=False)
    
    # 2. SATELLITE - position based on cross-track AND heading direction
    if sat_info is not None:
        sat_lon = sat_info['lon']
        sat_lat = sat_info['lat']
        sat_alt = sat_info['altitude']
        xtrk = sat_info['cross_track']
        
        # Position based on heading and cross-track
        # Heading 0-90 or 270-360: northward component
        # Heading 90-270: southward component
        if (0 <= heading < 90) or (270 <= heading < 360):
            # Northward: xtrk>0 means sat is on LEFT (west)
            if xtrk >= 0:
                sat_x, sat_y = -0.12, 0.5  # LEFT
            else:
                sat_x, sat_y = 1.12, 0.5   # RIGHT
        else:  # 90 <= heading < 270
            # Southward: xtrk>0 means sat is on RIGHT (east)
            if xtrk >= 0:
                sat_x, sat_y = 1.12, 0.5   # RIGHT
            else:
                sat_x, sat_y = -0.12, 0.5  # LEFT
        
        # SAT box
        ax.plot(sat_x, sat_y, 's', color='blue', markersize=12, transform=ax.transAxes,
                markeredgecolor='white', markeredgewidth=1, clip_on=False, zorder=15)
        ax.text(sat_x, sat_y, 'SAT', transform=ax.transAxes,
                fontsize=6, ha='center', va='center', color='white', fontweight='bold', clip_on=False, zorder=16)
        
        # Heading arrow FROM satellite box
        arrow_len = 0.12
        heading_rad = np.radians(heading)
        dx = arrow_len * np.sin(heading_rad)
        dy = arrow_len * np.cos(heading_rad)
        
        ax.annotate('', xy=(sat_x + dx, sat_y + dy), xytext=(sat_x, sat_y),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='blue', lw=1.5),
                    annotation_clip=False, zorder=14)
        
        # Heading label - position based on arrow direction
        label_x = sat_x + dx * 1.4
        label_y = sat_y + dy * 1.4
        ax.text(label_x, label_y, f'{heading:.0f}°', transform=ax.transAxes,
                ha='center', va='center', fontsize=8, fontweight='bold', color='blue', clip_on=False)
        
        # SAT info text - below the box with enough spacing
        info_text = f'Lon:{sat_lon:.2f}°\nLat:{sat_lat:.2f}°\nAlt:{sat_alt/1000:.0f}km\nXtrk:{xtrk/1000:.1f}km'
        ax.text(sat_x, sat_y - 0.10, info_text, transform=ax.transAxes,
                ha='center', va='top', fontsize=5, color='blue', clip_on=False,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='blue', alpha=0.9, linewidth=0.5),
                zorder=15)
    
    # 3. SUN - position based on azimuth
    if sun_azimuth is not None and sun_elevation is not None:
        if sun_elevation > 0:  # Daytime
            # Determine sun position based on azimuth
            # azimuth: 0=N, 90=E, 180=S, 270=W
            if 45 <= sun_azimuth < 135:  # East (right)
                sun_x, sun_y = 1.12, 0.85
                arrow_dx, arrow_dy = -0.08, 0
                label_x, label_y = sun_x + 0.08, sun_y  # Label to the right
                label_ha, label_va = 'left', 'center'
            elif 135 <= sun_azimuth < 225:  # South (bottom)
                sun_x, sun_y = 0.5, -0.12
                arrow_dx, arrow_dy = 0, 0.08
                label_x, label_y = sun_x, sun_y - 0.08  # Label below
                label_ha, label_va = 'center', 'top'
            elif 225 <= sun_azimuth < 315:  # West (left)
                sun_x, sun_y = -0.12, 0.85
                arrow_dx, arrow_dy = 0.08, 0
                label_x, label_y = sun_x - 0.08, sun_y  # Label to the left
                label_ha, label_va = 'right', 'center'
            else:  # North (top): 315-360 or 0-45
                sun_x, sun_y = 0.5, 1.12
                arrow_dx, arrow_dy = 0, -0.08
                label_x, label_y = sun_x, sun_y + 0.08  # Label above
                label_ha, label_va = 'center', 'bottom'
            
            # SUN box (orange)
            ax.plot(sun_x, sun_y, 's', color='#FFA500', markersize=12, transform=ax.transAxes,
                    markeredgecolor='white', markeredgewidth=1, clip_on=False, zorder=15)
            ax.text(sun_x, sun_y, 'SUN', transform=ax.transAxes,
                    fontsize=5, ha='center', va='center', color='white', fontweight='bold', clip_on=False, zorder=16)
            
            # Arrow FROM sun toward center
            ax.annotate('', xy=(sun_x + arrow_dx, sun_y + arrow_dy), xytext=(sun_x, sun_y),
                        xycoords='axes fraction', textcoords='axes fraction',
                        arrowprops=dict(arrowstyle='->', color='#FFA500', lw=1.5),
                        annotation_clip=False, zorder=14)
            
            # Sun Az and El label - positioned away from box
            sun_label = f'Az:{sun_azimuth:.0f}° El:{sun_elevation:.0f}°'
            ax.text(label_x, label_y, sun_label, transform=ax.transAxes, 
                    ha=label_ha, va=label_va,
                    fontsize=6, fontweight='bold', color='#FFA500', clip_on=False)
        else:
            # Night - just show text
            ax.text(1.06, 0.85, 'Night', transform=ax.transAxes,
                    ha='center', va='center', fontsize=8, color='#555555', clip_on=False)

# Classification colors
CLASSIFICATION_COLORS = [
    'maroon',        # 1: land - 고동색
    'gold',          # 2: land_near_water - 진한 노랑색
    'skyblue',       # 3: water_near_land - 하늘색
    'mediumblue',    # 4: open_water - 새파란색
    'dimgray',       # 5: dark_water - 진한 회색
    'cyan',          # 6: low_coh_water_near_land - 옅은 하늘색
    'darkturquoise', # 7: open_low_coh_water - 옅은 새파란색
]

CLASSIFICATION_LABELS = [
    'land', 'land_near_water', 'water_near_land', 'open_water',
    'dark_water', 'low_coh_water_near_land', 'open_low_coh_water'
]

POLYGON_COLOR = '#404040'

def process_fctimp(fctimp_str, spike_threshold=1.0, output_dir=None):
    if output_dir is None:
        output_dir = f'isa_{fctimp_str}_v2'
    Path(output_dir).mkdir(exist_ok=True)
    
    selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
    fctimp_val = float(fctimp_str)
    sample = selected_12[np.isclose(selected_12["fctimp_mea"], fctimp_val, atol=0.01)].copy()
    if sample.empty:
        print(f"No reach for {fctimp_str}")
        return
    
    df = pd.read_parquet(f'/work/pi_kandread_umass_edu/swot-urban/1_data_processing/fctimp_mean_{fctimp_str}_sample.parquet')
    d = df.copy()
    d['time_dt'] = pd.to_datetime(d['time_str'])
    d = d.sort_values('time_dt').reset_index(drop=True)
    d['error'] = d['wse_relative_m'] - d['wse_usgs_relative_m']  # + = overestimate, - = underestimate
    d['is_spike'] = d['error'] >= spike_threshold
    
    pixc_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXC/fctimp_{fctimp_str}')
    pixcvec_dir = Path(f'/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_PIXCVec/fctimp_{fctimp_str}')
    pixc_files = filter_best_pixc_files(list(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc')))
    pixcvec_files = filter_best_pixc_files(list(pixcvec_dir.glob('SWOT_L2_HR_PIXCVec_*.nc')))
    
    polygon = sample.to_crs('EPSG:4326')
    bounds = polygon.total_bounds
    
    for idx, spike_row in d[d['is_spike']].iterrows():
        print(f"\n{'='*70}")
        print(f"Spike: {spike_row['time_str']} (error={spike_row['error']:.2f}m)")
        
        normals = d[~d['is_spike']].copy()
        if len(normals) == 0:
            continue
        normals['time_diff'] = (normals['time_dt'] - spike_row['time_dt']).abs()
        closest_normal = normals.loc[normals['time_diff'].idxmin()]
        print(f"Normal: {closest_normal['time_str']} (error={closest_normal['error']:.2f}m, Δ={closest_normal['time_diff']})")
        
        # Get Normal data
        normal_data = None
        normal_info = None
        for group_key, nf_list in match_and_group_pixc(closest_normal['time_str'], pixc_files, closest_normal['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(nf, pixcvec_files) for nf in nf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_pixels(nf_list, vec_list, polygon)
            if data:
                normal_data = data
                normal_info = {
                    'time_str': closest_normal['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': closest_normal['time_dt'],
                    'crid': closest_normal['crid'],
                    'error': closest_normal['error'],
                    'tile': get_tile_from_filename(nf_list[0].name),
                    'heading': data['heading'],
                    'group_key': group_key,
                    'sat_info': data['sat_info']
                }
                break
        
        # Get Spike data
        spike_data = None
        spike_info = None
        for group_key, sf_list in match_and_group_pixc(spike_row['time_str'], pixc_files, spike_row['crid']).items():
            vec_list = [v for v in [find_matching_pixcvec(sf, pixcvec_files) for sf in sf_list] if v]
            if not vec_list:
                continue
            data = extract_and_merge_pixels(sf_list, vec_list, polygon)
            if data:
                spike_data = data
                spike_info = {
                    'time_str': spike_row['time_dt'].strftime('%Y%m%dT%H%M%S'),
                    'time_dt': spike_row['time_dt'],
                    'crid': spike_row['crid'],
                    'error': spike_row['error'],
                    'tile': get_tile_from_filename(sf_list[0].name),
                    'heading': data['heading'],
                    'group_key': group_key,
                    'sat_info': data['sat_info']
                }
                break
        
        # Plot Normal-Spike comparison if both exist
        if normal_data and spike_data:
            save_comparison_plots(normal_data, spike_data, normal_info, spike_info, 
                                  polygon, bounds, output_dir, fctimp_str)
    
    print(f"\n✓ Done: {output_dir}")

def match_and_group_pixc(time_str, files, crid, window_minutes=5):
    dt = pd.to_datetime(time_str)
    if dt.tzinfo:
        dt = dt.tz_localize(None)
    start, end = dt - timedelta(minutes=window_minutes), dt + timedelta(minutes=window_minutes)
    groups = {}
    for f in files:
        parts = f.name.split('_')
        if len(parts) >= 10:
            try:
                fs, fe = pd.to_datetime(parts[7], format='%Y%m%dT%H%M%S'), pd.to_datetime(parts[8], format='%Y%m%dT%H%M%S')
                if not (fe < start or fs > end) and parts[9] == crid:
                    key = f"{parts[7]}_{parts[8]}"
                    groups.setdefault(key, []).append(f)
            except:
                continue
    return groups

def find_matching_pixcvec(pixc_file, pixcvec_files):
    pixc_name = pixc_file.name.replace('PIXC_', 'PIXCVec_')
    for pvf in pixcvec_files:
        if pvf.name == pixc_name:
            return pvf
    return None

def extract_and_merge_pixels(pixc_files, pixcvec_files, polygon_gdf):
    all_data = [extract_pixels_with_vec(pf, pvf, polygon_gdf) for pf, pvf in zip(pixc_files, pixcvec_files)]
    all_data = [d for d in all_data if d]
    if not all_data:
        return None
    if len(all_data) == 1:
        return all_data[0]
    merged = {k: np.concatenate([d[k] for d in all_data]) for k in all_data[0].keys() 
              if k not in ['heading', 'sat_info']}
    merged['heading'] = np.nanmean([d['heading'] for d in all_data])
    # Use first satellite info (they should be very similar)
    merged['sat_info'] = all_data[0]['sat_info']
    return merged

def extract_pixels_with_vec(pixc_file, pixcvec_file, polygon_gdf):
    ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    lon, lat = ds['longitude'].values, ds['latitude'].values
    bounds = polygon_gdf.total_bounds
    
    # Get tvp group data (satellite position)
    ds_tvp = xr.open_dataset(pixc_file, group='tvp')
    heading = np.nanmedian(ds_tvp['velocity_heading'].values)
    
    # Satellite position - use median values
    sat_lon = np.nanmedian(ds_tvp['longitude'].values)
    sat_lat = np.nanmedian(ds_tvp['latitude'].values)
    sat_alt = np.nanmedian(ds_tvp['altitude'].values)
    ds_tvp.close()
    
    bbox_mask = ((lon >= bounds[0]) & (lon <= bounds[2]) & (lat >= bounds[1]) & (lat <= bounds[3]))
    if bbox_mask.sum() == 0:
        ds.close()
        return None
    
    classification = ds['classification'].values[bbox_mask]
    water_mask = classification > 2
    if water_mask.sum() == 0:
        ds.close()
        return None
    
    lon_filt, lat_filt = lon[bbox_mask][water_mask], lat[bbox_mask][water_mask]
    poly = polygon_gdf.geometry.iloc[0]
    inside_mask = np.array([poly.contains(Point(x, y)) for x, y in zip(lon_filt, lat_filt)])
    if inside_mask.sum() == 0:
        ds.close()
        return None
    
    final_idx = np.where(bbox_mask)[0][water_mask][inside_mask]
    
    # Get cross-track distance
    cross_track = ds['cross_track'].values[final_idx]
    median_cross_track = np.nanmedian(cross_track)
    
    ds_vec = xr.open_dataset(pixcvec_file)
    reach_id = ds_vec['reach_id'].values[final_idx]
    node_id = ds_vec['node_id'].values[final_idx]
    obs_id = ds_vec['obs_id'].values[final_idx]
    lake_id = ds_vec['lake_id'].values[final_idx]
    
    def is_valid(arr):
        return np.array([len(x) > 0 and x != b'' for x in arr]) if arr.dtype.kind == 'S' else arr > 0
    
    selected_mask = is_valid(reach_id) & is_valid(node_id) & ~is_valid(obs_id) & ~is_valid(lake_id)
    
    result = {
        'lon': lon[final_idx], 'lat': lat[final_idx],
        'lon_all': lon, 'lat_all': lat,
        'height': ds['height'].values[final_idx],
        'classification': ds['classification'].values[final_idx],
        'bright_land': ds['bright_land_flag'].values[final_idx],
        'layover': ds['layover_impact'].values[final_idx],
        'sig0': ds['sig0'].values[final_idx],
        'coh_power': ds['coherent_power'].values[final_idx],
        'azimuth': ds['azimuth_index'].values[final_idx],
        'range': ds['range_index'].values[final_idx],
        'cross_track': cross_track,
        'selected_mask': selected_mask,
        'heading': heading,
        'sat_info': {
            'lon': sat_lon,
            'lat': sat_lat,
            'altitude': sat_alt,
            'cross_track': median_cross_track
        }
    }
    ds.close()
    ds_vec.close()
    return result

def save_comparison_plots(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, fctimp_str):
    """Save side-by-side Normal vs Spike comparison plots"""
    
    time_str = spike_info['time_str']
    
    # Calculate sun positions
    poly_center_lat = (bounds[1] + bounds[3]) / 2
    poly_center_lon = (bounds[0] + bounds[2]) / 2
    
    normal_sun_az, normal_sun_el = calculate_sun_position(
        normal_info['time_dt'].to_pydatetime() if hasattr(normal_info['time_dt'], 'to_pydatetime') else normal_info['time_dt'],
        poly_center_lat, poly_center_lon
    )
    spike_sun_az, spike_sun_el = calculate_sun_position(
        spike_info['time_dt'].to_pydatetime() if hasattr(spike_info['time_dt'], 'to_pydatetime') else spike_info['time_dt'],
        poly_center_lat, poly_center_lon
    )
    
    normal_info['sun_azimuth'] = normal_sun_az
    normal_info['sun_elevation'] = normal_sun_el
    spike_info['sun_azimuth'] = spike_sun_az
    spike_info['sun_elevation'] = spike_sun_el
    
    # 1. Combined comparison (now 2x4 with sig0)
    plot_combined_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str)
    
    # 2. Height comparison
    plot_height_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str)
    
    # 3. Classification comparison
    plot_classification_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str)
    
    # 4. Sig0 comparison (NEW)
    plot_sig0_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str)
    
    # 5. Statistics comparison
    plot_statistics_comparison(normal_data, spike_data, normal_info, spike_info, output_dir, time_str, fctimp_str)
    
    print(f"  Saved: {time_str}")

def plot_combined_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str):
    """2x4 grid: Top row = Normal, Bottom row = Spike (Height, Classification, Sig0, NLCD)"""
    
    dx = bounds[2] - bounds[0]
    dy = bounds[3] - bounds[1]
    aspect = dy / dx
    
    panel_width = 2.5
    fig_width = panel_width * 4 + 1.2  # 4 columns now
    fig_height = panel_width * aspect * 2 + 0.5
    
    fig, axes = plt.subplots(2, 4, figsize=(fig_width, fig_height), constrained_layout=True)
    
    nlcd_path = '/work/pi_kandread_umass_edu/swot-urban/0_data_collection/1_nlcd/Annual_NLCD_LndCov_2024_CU_C1V1_EPSG4326.tif'
    nlcd_colors = {11: '#476BA1', 12: '#D1DDF9', 21: '#DDC9C9', 22: '#D89382', 23: '#ED0000', 24: '#AA0000',
                   31: '#B2ADA3', 41: '#68AA63', 42: '#1C6330', 43: '#B5C98E', 52: '#CCBA7C', 71: '#E2E2C1',
                   81: '#DDD83E', 82: '#AE7229', 90: '#BBD7ED', 95: '#71A4C1'}
    
    for row, (data, info, label) in enumerate([
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        unsel = ~data['selected_mask']
        sel = data['selected_mask']
        heading = info['heading']
        tile = info['tile']
        error = info['error']
        sun_az = info['sun_azimuth']
        sun_el = info['sun_elevation']
        
        # Column 0: Height
        ax = axes[row, 0]
        if sel.sum() > 0:
            ax.scatter(data['lon'][sel], data['lat'][sel], c=data['height'][sel],
                       cmap='viridis', s=0.15, alpha=0.7, marker='x', linewidths=0.15)
        if unsel.sum() > 0:
            ax.scatter(data['lon'][unsel], data['lat'][unsel], c=data['height'][unsel],
                       cmap='viridis', s=0.15, alpha=0.4, marker='o', edgecolors='none')

        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_title(f'{label} | {heading:.0f}°-{tile}\nError: {error:.2f}m', fontsize=10, fontweight='bold',
                     color='green' if label == 'NORMAL' else 'red')
        
        # Column 1: Classification
        ax = axes[row, 1]
        if sel.sum() > 0:
            ax.scatter(data['lon'][sel], data['lat'][sel], c=data['classification'][sel],
                       cmap=ListedColormap(CLASSIFICATION_COLORS), s=0.15, alpha=0.7, marker='x', linewidths=0.15, vmin=1, vmax=7)
        if unsel.sum() > 0:
            ax.scatter(data['lon'][unsel], data['lat'][unsel], c=data['classification'][unsel],
                       cmap=ListedColormap(CLASSIFICATION_COLORS), s=0.15, alpha=0.4, marker='o', vmin=1, vmax=7, edgecolors='none')

        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_title('Classification', fontsize=10)
        
        # Column 2: Sig0 (NEW)
        ax = axes[row, 2]
        if sel.sum() > 0:
            ax.scatter(data['lon'][sel], data['lat'][sel], c=data['sig0'][sel],
                       cmap='cool', s=0.15, alpha=0.7, marker='x', linewidths=0.15)
        if unsel.sum() > 0:
            ax.scatter(data['lon'][unsel], data['lat'][unsel], c=data['sig0'][unsel],
                       cmap='cool', s=0.15, alpha=0.4, marker='o', edgecolors='none')

        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_title('Sig0 (σ₀)', fontsize=10)
        
        # Column 3: NLCD
        ax = axes[row, 3]
        buffer = 0.005
        with rasterio.open(nlcd_path) as src:
            window = src.window(bounds[0]-buffer, bounds[1]-buffer, bounds[2]+buffer, bounds[3]+buffer)
            nlcd_data = src.read(1, window=window)
            unique_vals = np.unique(nlcd_data[nlcd_data > 0])
            cmap_list = [nlcd_colors.get(v, '#FFFFFF') for v in range(int(unique_vals.min()), int(unique_vals.max())+1)]
            extent = [bounds[0]-buffer, bounds[2]+buffer, bounds[1]-buffer, bounds[3]+buffer]
            ax.imshow(nlcd_data, extent=extent, cmap=ListedColormap(cmap_list),
                      vmin=unique_vals.min(), vmax=unique_vals.max(), interpolation='nearest')
        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_title('NLCD', fontsize=10)
        
        # Common settings
        for col, ax in enumerate(axes[row]):
            ax.set_xlim(bounds[0], bounds[2])
            ax.set_ylim(bounds[1], bounds[3])
            ax.set_aspect('equal')
            ax.axis('off')
            # Only add arrows and satellite info to NLCD panel (column 3)
            if col == 3:
                add_arrows_and_indicators(ax, bounds, heading, sun_az, sun_el, info['sat_info'])
    
    fig.suptitle(f'ISA={fctimp_str} | Normal vs Spike Comparison\n(X=selected, O=unselected)', 
                 fontsize=11, fontweight='bold')
    plt.savefig(f'{output_dir}/COMPARE_{time_str}_Combined.png', dpi=1200, bbox_inches='tight', pad_inches=0.15)
    plt.close()

def plot_height_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str):
    """Side-by-side height comparison"""
    
    dx = bounds[2] - bounds[0]
    dy = bounds[3] - bounds[1]
    aspect = dy / dx
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6 * aspect))
    
    for ax, (data, info, label) in zip(axes, [
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        unsel = ~data['selected_mask']
        sel = data['selected_mask']
        
        if sel.sum() > 0:
            sc = ax.scatter(data['lon'][sel], data['lat'][sel], c=data['height'][sel],
                           cmap='viridis', s=0.15, alpha=0.7, marker='x', linewidths=0.15)
            plt.colorbar(sc, ax=ax, label='Height (m)', shrink=0.7)
            
        if unsel.sum() > 0:
            ax.scatter(data['lon'][unsel], data['lat'][unsel], c=data['height'][unsel],
                       cmap='viridis', s=0.15, alpha=0.4, marker='o', edgecolors='none')
        
        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_xlim(bounds[0], bounds[2])
        ax.set_ylim(bounds[1], bounds[3])
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f'{label} | {info["heading"]:.0f}°-{info["tile"]}\nError: {info["error"]:.2f}m', 
                     fontsize=11, fontweight='bold', color='green' if label == 'NORMAL' else 'red')
    
    fig.suptitle(f'ISA={fctimp_str} | Height Comparison', fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(f'{output_dir}/COMPARE_{time_str}_Height.png', dpi=600, bbox_inches='tight')
    plt.close()

def plot_classification_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str):
    """Side-by-side classification comparison"""
    
    dx = bounds[2] - bounds[0]
    dy = bounds[3] - bounds[1]
    aspect = dy / dx
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6 * aspect))
    
    for ax, (data, info, label) in zip(axes, [
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        unsel = ~data['selected_mask']
        sel = data['selected_mask']
        if sel.sum() > 0:
            ax.scatter(data['lon'][sel], data['lat'][sel], c=data['classification'][sel],
                       cmap=ListedColormap(CLASSIFICATION_COLORS), s=0.15, alpha=0.7, marker='x', linewidths=0.15, vmin=1, vmax=7)
            
        if unsel.sum() > 0:
            ax.scatter(data['lon'][unsel], data['lat'][unsel], c=data['classification'][unsel],
                       cmap=ListedColormap(CLASSIFICATION_COLORS), s=0.15, alpha=0.4, marker='o', vmin=1, vmax=7, edgecolors='none')

        
        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_xlim(bounds[0], bounds[2])
        ax.set_ylim(bounds[1], bounds[3])
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f'{label} | {info["heading"]:.0f}°-{info["tile"]}\nError: {info["error"]:.2f}m', 
                     fontsize=11, fontweight='bold', color='green' if label == 'NORMAL' else 'red')
    
    legend = [Patch(facecolor=CLASSIFICATION_COLORS[i], label=CLASSIFICATION_LABELS[i]) for i in range(7)]
    fig.legend(handles=legend, loc='center right', bbox_to_anchor=(0.99, 0.5), fontsize=9)
    
    fig.suptitle(f'ISA={fctimp_str} | Classification Comparison', fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 0.88, 0.94])
    plt.savefig(f'{output_dir}/COMPARE_{time_str}_Classification.png', dpi=600, bbox_inches='tight')
    plt.close()

def plot_sig0_comparison(normal_data, spike_data, normal_info, spike_info, polygon, bounds, output_dir, time_str, fctimp_str):
    """Side-by-side sig0 (radar backscatter) comparison"""
    
    dx = bounds[2] - bounds[0]
    dy = bounds[3] - bounds[1]
    aspect = dy / dx
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6 * aspect))
    
    # Get common vmin/vmax for both plots
    all_sig0 = np.concatenate([
        normal_data['sig0'][normal_data['selected_mask']],
        spike_data['sig0'][spike_data['selected_mask']]
    ])
    # Filter out invalid values
    valid_sig0 = all_sig0[np.isfinite(all_sig0)]
    if len(valid_sig0) > 0:
        vmin = np.percentile(valid_sig0, 2)
        vmax = np.percentile(valid_sig0, 98)
    else:
        vmin, vmax = None, None
    
    for ax, (data, info, label) in zip(axes, [
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        unsel = ~data['selected_mask']
        sel = data['selected_mask']
        if sel.sum() > 0:
            sc = ax.scatter(data['lon'][sel], data['lat'][sel], c=data['sig0'][sel],
                           cmap='cool', s=0.15, alpha=0.7, marker='x', linewidths=0.15,
                           vmin=vmin, vmax=vmax)
            plt.colorbar(sc, ax=ax, label='Sig0 (dB)', shrink=0.7)
            
        if unsel.sum() > 0:
            ax.scatter(data['lon'][unsel], data['lat'][unsel], c=data['sig0'][unsel],
                       cmap='cool', s=0.15, alpha=0.4, marker='o', edgecolors='none',
                       vmin=vmin, vmax=vmax)

        
        polygon.boundary.plot(ax=ax, color=POLYGON_COLOR, linewidth=2)
        ax.set_xlim(bounds[0], bounds[2])
        ax.set_ylim(bounds[1], bounds[3])
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f'{label} | {info["heading"]:.0f}°-{info["tile"]}\nError: {info["error"]:.2f}m', 
                     fontsize=11, fontweight='bold', color='green' if label == 'NORMAL' else 'red')
    
    fig.suptitle(f'ISA={fctimp_str} | Sig0 (σ₀) Comparison', fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(f'{output_dir}/COMPARE_{time_str}_Sig0.png', dpi=600, bbox_inches='tight')
    plt.close()

def plot_statistics_comparison(normal_data, spike_data, normal_info, spike_info, output_dir, time_str, fctimp_str):
    """Side-by-side statistics comparison"""
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    
    for row, (data, info, label) in enumerate([
        (normal_data, normal_info, 'NORMAL'),
        (spike_data, spike_info, 'SPIKE')
    ]):
        sel = data['selected_mask']
        color = 'green' if label == 'NORMAL' else 'red'
        
        # Classification distribution
        axes[row, 0].bar(range(1, 8), [np.sum(data['classification'][sel] == i) for i in range(1, 8)], color=color, alpha=0.7)
        axes[row, 0].set_xlabel('Classification')
        axes[row, 0].set_ylabel('Count')
        axes[row, 0].set_title(f'{label}: Classification')
        axes[row, 0].grid(alpha=0.3)
        
        # Bright land
        axes[row, 1].bar(range(2), [np.sum(data['bright_land'][sel] == i) for i in range(2)], color=color, alpha=0.7)
        axes[row, 1].set_xlabel('Bright Land')
        axes[row, 1].set_ylabel('Count')
        axes[row, 1].set_title(f'{label}: Bright Land')
        axes[row, 1].set_xticks([0, 1])
        axes[row, 1].grid(alpha=0.3)
        
        # Boxplots
        if sel.sum() > 0:
            for col, key in zip([2, 3, 4], ['layover', 'sig0', 'coh_power']):
                bp = axes[row, col].boxplot(data[key][sel], vert=True, patch_artist=True)
                bp['boxes'][0].set_facecolor(color)
                bp['boxes'][0].set_alpha(0.5)
                axes[row, col].set_ylabel(key.replace('_', ' ').title())
                axes[row, col].set_title(f'{label}: {key.replace("_", " ").title()}')
                axes[row, col].grid(alpha=0.3)
    
    fig.suptitle(f'ISA={fctimp_str} | Statistics Comparison\nNormal (Error={normal_info["error"]:.2f}m) vs Spike (Error={spike_info["error"]:.2f}m)', 
                 fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(f'{output_dir}/COMPARE_{time_str}_Statistics.png', dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# ============================================================
# FULL
# ============================================================
for fc in ['0.01', '0.06', '0.12', '0.17', '0.22', '0.32', '0.28', '0.40', '0.45', '0.50', '0.55', '0.61']:
    process_fctimp(fc, spike_threshold=0.1832)

### NO ZOOM

In [ ]:
# import xarray as xr
# import matplotlib.pyplot as plt
# import geopandas as gpd
# from pathlib import Path
# import pandas as pd

# # 시간 순으로 정렬
# d_sorted = d.sort_values('time_dt').reset_index(drop=True)
# d_sorted['error'] = (d_sorted['wse_relative_m'] - d_sorted['wse_usgs_relative_m']).abs()
# d_sorted['is_spike'] = d_sorted['error'] >= 1.0

# pixc_dir = Path('/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_pixel_001')
# pixc_files = list(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc'))
# polygon = sample_1.to_crs('EPSG:4326')

# def match_pixc_file(time_str, files):
#     # time_str format: 2023-12-09 02:43:44
#     dt = pd.to_datetime(time_str)
#     date_time = dt.strftime('%Y%m%dT%H%M%S')  # 20231209T024344
    
#     for f in files:
#         if date_time[:12] in f.name:  # 분까지만 매칭 (초는 다를 수 있음)
#             return f
#     return None

# def plot_pixc_with_polygon(pixc_file, polygon, title, label):
#     ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    
#     fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
#     from matplotlib.colors import ListedColormap
#     colors_7 = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
#     cmap_class = ListedColormap(colors_7)
    
#     # Height
#     scatter1 = axes[0].scatter(ds['longitude'], ds['latitude'], 
#                               c=ds['height'], cmap='viridis', 
#                               s=0.0005, alpha=0.7)
#     polygon.boundary.plot(ax=axes[0], color='red', linewidth=2, alpha=0.5)
#     axes[0].set_xlabel('Longitude')
#     axes[0].set_ylabel('Latitude')
#     axes[0].set_title(f'{label} - {title}\nPixel Cloud Height')
#     plt.colorbar(scatter1, ax=axes[0], label='Height (m)')
    
#     # Classification
#     scatter2 = axes[1].scatter(ds['longitude'], ds['latitude'],
#                               c=ds['classification'], cmap=cmap_class,
#                               s=0.0005, alpha=0.7, vmin=1, vmax=7)
#     polygon.boundary.plot(ax=axes[1], color='red', linewidth=2, alpha=0.5)
#     axes[1].set_xlabel('Longitude')
#     axes[1].set_ylabel('Latitude')
#     axes[1].set_title(f'{label} - {title}\nClassification')
#     plt.colorbar(scatter2, ax=axes[1], label='Class', ticks=[1,2,3,4,5,6,7])
    
#     plt.tight_layout()
#     return fig

# # 각 spike에 대해 직전 normal 찾기
# spikes = d_sorted[d_sorted['is_spike']].copy()
# isa_r = spikes.fctimp_mean.iloc[0].round(2)

# for idx, spike_row in spikes.iterrows():
#     spike_time = spike_row['time_dt']
#     spike_date = spike_time.strftime('%Y%m%dT%H%M%S')
    
#     # 직전 normal 찾기
#     previous_normals = d_sorted[(d_sorted['time_dt'] < spike_time) & (~d_sorted['is_spike'])]
    
#     if len(previous_normals) > 0:
#         prev_normal = previous_normals.iloc[-1]
#         prev_date = prev_normal['time_dt'].strftime('%Y%m%dT%H%M%S')
        
#         print(f"\n{'='*70}")
#         print(f"Spike: {spike_date} (error={spike_row['error']:.2f}m)")
#         print(f"Previous Normal: {prev_date} (error={prev_normal['error']:.2f}m)")
        
#         # Plot previous normal
#         pixc_file = match_pixc_file(prev_date, pixc_files)
#         if pixc_file:
#             fig = plot_pixc_with_polygon(pixc_file, polygon, prev_date, "NORMAL (Before Spike)")
#             plt.savefig(f'isa_001/pixc_normal_before_{spike_date}_{isa_r}.png', dpi=500, bbox_inches='tight')
#             plt.show()
        
#         # Plot spike
#         pixc_file = match_pixc_file(spike_date, pixc_files)
#         if pixc_file:
#             fig = plot_pixc_with_polygon(pixc_file, polygon, spike_date, "SPIKE")
#             plt.savefig(f'isa_001/pixc_spike_{spike_date}_{isa_r}.png', dpi=500, bbox_inches='tight')
#             plt.show()

### ZOOM

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import timedelta
from shapely.vectorized import contains

d_sorted = d.sort_values('time_dt').reset_index(drop=True)
d_sorted['error'] = (d_sorted['wse_relative_m'] - d_sorted['wse_usgs_relative_m']).abs()
d_sorted['is_spike'] = d_sorted['error'] >= 0.1832

pixc_dir = Path('/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_pixel_001')
pixc_files = list(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc'))
polygon = sample_1.to_crs('EPSG:4326')
bounds = polygon.total_bounds

def match_pixc_files(time_str, files, crid, window_minutes=5):
    dt = pd.to_datetime(time_str)
    if dt.tzinfo is not None:
        dt = dt.tz_localize(None)
    start_time = dt - timedelta(minutes=window_minutes)
    end_time = dt + timedelta(minutes=window_minutes)
    
    matched = []
    for f in files:
        parts = f.name.split('_')
        if len(parts) >= 10:
            try:
                file_start = pd.to_datetime(parts[7], format='%Y%m%dT%H%M%S')
                file_end = pd.to_datetime(parts[8], format='%Y%m%dT%H%M%S')
                file_crid = parts[9]
                tile_info = parts[6]
                if (not (file_end < start_time or file_start > end_time)) and (file_crid == crid):
                    matched.append((f, tile_info))
            except:
                continue
    return matched

def extract_pixels(pixc_file, polygon_gdf):
    ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    lon = ds['longitude'].values
    lat = ds['latitude'].values
    bounds = polygon_gdf.total_bounds
    
    bbox_mask = ((lon >= bounds[0]) & (lon <= bounds[2]) & 
                 (lat >= bounds[1]) & (lat <= bounds[3]))
    if bbox_mask.sum() == 0:
        ds.close()
        return None
    
    classification = ds['classification'].values[bbox_mask]
    water_mask = classification > 2
    if water_mask.sum() == 0:
        ds.close()
        return None
    
    lon_filt = lon[bbox_mask][water_mask]
    lat_filt = lat[bbox_mask][water_mask]
    poly = polygon_gdf.geometry.iloc[0]
    inside_mask = contains(poly, lon_filt, lat_filt)
    if inside_mask.sum() == 0:
        ds.close()
        return None
    
    final_idx = np.where(bbox_mask)[0][water_mask][inside_mask]
    result = {
        'lon': lon[final_idx],
        'lat': lat[final_idx],
        'lon_all': lon,
        'lat_all': lat,
        'height': ds['height'].values[final_idx],
        'classification': ds['classification'].values[final_idx],
        'bright_land': ds['bright_land_flag'].values[final_idx],
        'layover': ds['layover_impact'].values[final_idx],
        'sig0': ds['sig0'].values[final_idx],
        'coh_power': ds['coherent_power'].values[final_idx],
        'azimuth': ds['azimuth_index'].values[final_idx],
        'range': ds['range_index'].values[final_idx]
    }
    ds.close()
    return result

def plot_height(data, poly, label):
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(data['lon'], data['lat'], c=data['height'], 
                    cmap='viridis', s=0.01, alpha=0.7)
    poly.boundary.plot(ax=ax, color='red', linewidth=2)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'{label} - Height')
    plt.colorbar(sc, ax=ax, label='Height (m)')
    plt.tight_layout()
    return fig

def plot_classification(data, poly, label):
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ['#8B4513', '#FFA500', '#87CEEB', '#FF0000', '#000080', '#808080', '#4169E1']
    labels = ['land', 'land_near_water', 'water_near_land', 'open_water', 
              'dark_water', 'low_coh_water_near_land', 'open_low_coh_water']
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch
    
    sc = ax.scatter(data['lon'], data['lat'], c=data['classification'], 
                    cmap=ListedColormap(colors), s=0.01, alpha=0.8, vmin=1, vmax=7)
    poly.boundary.plot(ax=ax, color='black', linewidth=2)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'{label} - Classification')
    
    legend = [Patch(facecolor=colors[i], label=labels[i]) for i in range(7)]
    ax.legend(handles=legend, loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)
    plt.tight_layout()
    return fig

def plot_tile_overview(data, poly, label):
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(data['lon_all'], data['lat_all'], c='lightgray', s=0.1, alpha=0.3)
    ax.scatter(data['lon'], data['lat'], c='blue', s=0.01, alpha=0.5)
    poly.boundary.plot(ax=ax, color='red', linewidth=3)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'{label} - Tile Overview with Polygon')
    plt.tight_layout()
    return fig

def plot_statistics(data, label):
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    # 1. Classification count
    class_counts = [np.sum(data['classification'] == i) for i in range(1, 8)]
    axes[0].bar(range(1, 8), class_counts, color='steelblue')
    axes[0].set_xlabel('Classification')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Classification Count')
    axes[0].grid(alpha=0.3)
    
    # 2. Bright land flag count
    bright_counts = [np.sum(data['bright_land'] == i) for i in range(2)]
    axes[1].bar(range(2), bright_counts, color='orange')
    axes[1].set_xlabel('Bright Land Flag')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Bright Land Flag Count')
    axes[1].set_xticks([0, 1])
    axes[1].grid(alpha=0.3)
    
    # 3. Layover impact boxplot
    axes[2].boxplot(data['layover'], vert=True)
    axes[2].set_ylabel('Layover Impact')
    axes[2].set_title('Layover Impact')
    axes[2].grid(alpha=0.3)
    
    # 4. Sigma0 boxplot
    axes[3].boxplot(data['sig0'], vert=True)
    axes[3].set_ylabel('Sigma0 (dB)')
    axes[3].set_title('Sigma0')
    axes[3].grid(alpha=0.3)
    
    # 5. Coherent power boxplot
    axes[4].boxplot(data['coh_power'], vert=True)
    axes[4].set_ylabel('Coherent Power')
    axes[4].set_title('Coherent Power')
    axes[4].grid(alpha=0.3)
    
    # 6. Total pixel count
    axes[5].bar([0], [len(data['lon'])], color='green')
    axes[5].set_ylabel('Count')
    axes[5].set_title(f'Total Pixels: {len(data["lon"])}')
    axes[5].set_xticks([])
    axes[5].grid(alpha=0.3)
    
    # 7. Azimuth boxplot
    axes[6].boxplot(data['azimuth'], vert=True)
    axes[6].set_ylabel('Azimuth Index')
    axes[6].set_title('Azimuth Index')
    axes[6].grid(alpha=0.3)
    
    # 8. Range boxplot
    axes[7].boxplot(data['range'], vert=True)
    axes[7].set_ylabel('Range Index')
    axes[7].set_title('Range Index')
    axes[7].grid(alpha=0.3)
    
    fig.suptitle(label, fontsize=14, y=1.00)
    plt.tight_layout()
    return fig

# Main loop
spikes = d_sorted[d_sorted['is_spike']].copy()
isa_r = spikes.fctimp_mean.iloc[0].round(2)

for idx, spike_row in spikes.iterrows():
    print(f"\n{'='*70}")
    print(f"Spike: {spike_row['time_str']} (error={spike_row['error']:.2f}m, crid={spike_row['crid']})")
    
    previous_normals = d_sorted[(d_sorted['time_dt'] < spike_row['time_dt']) & (~d_sorted['is_spike'])]
    if len(previous_normals) == 0:
        continue
    
    prev_normal = previous_normals.iloc[-1]
    print(f"Normal: {prev_normal['time_str']} (error={prev_normal['error']:.2f}m, crid={prev_normal['crid']})")
    
    # Normal files
    normal_files = match_pixc_files(prev_normal['time_str'], pixc_files, prev_normal['crid'])
    for nf, tile in normal_files:
        data = extract_pixels(nf, polygon)
        if data:
            time_str = prev_normal['time_dt'].strftime('%Y%m%dT%H%M%S')
            crid = prev_normal['crid']
            
            fig = plot_height(data, polygon, f'NORMAL-{time_str}-{crid}-{tile}')
            plt.savefig(f'isa_001/NORMAL_{time_str}_{crid}_{tile}_Height.png', dpi=300, bbox_inches='tight')
            plt.close()
            
            fig = plot_classification(data, polygon, f'NORMAL-{time_str}-{crid}-{tile}')
            plt.savefig(f'isa_001/NORMAL_{time_str}_{crid}_{tile}_Classification.png', dpi=300, bbox_inches='tight')
            plt.close()
            
            fig = plot_tile_overview(data, polygon, f'NORMAL-{time_str}-{crid}-{tile}')
            plt.savefig(f'isa_001/NORMAL_{time_str}_{crid}_{tile}_Overview.png', dpi=300, bbox_inches='tight')
            plt.close()
            
            fig = plot_statistics(data, f'NORMAL-{time_str}-{crid}-{tile}')
            plt.savefig(f'isa_001/NORMAL_{time_str}_{crid}_{tile}_Statistics.png', dpi=300, bbox_inches='tight')
            plt.close()
    
    # Spike files
    spike_files = match_pixc_files(spike_row['time_str'], pixc_files, spike_row['crid'])
    for sf, tile in spike_files:
        data = extract_pixels(sf, polygon)
        if data:
            time_str = spike_row['time_dt'].strftime('%Y%m%dT%H%M%S')
            crid = spike_row['crid']
            
            fig = plot_height(data, polygon, f'SPIKE-{time_str}-{crid}-{tile}')
            plt.savefig(f'isa_001/SPIKE_{time_str}_{crid}_{tile}_Height.png', dpi=300, bbox_inches='tight')
            plt.close()
            
            fig = plot_classification(data, polygon, f'SPIKE-{time_str}-{crid}-{tile}')
            plt.savefig(f'isa_001/SPIKE_{time_str}_{crid}_{tile}_Classification.png', dpi=300, bbox_inches='tight')
            plt.close()
            
            fig = plot_tile_overview(data, polygon, f'SPIKE-{time_str}-{crid}-{tile}')
            plt.savefig(f'isa_001/SPIKE_{time_str}_{crid}_{tile}_Overview.png', dpi=300, bbox_inches='tight')
            plt.close()
            
            fig = plot_statistics(data, f'SPIKE-{time_str}-{crid}-{tile}')
            plt.savefig(f'isa_001/SPIKE_{time_str}_{crid}_{tile}_Statistics.png', dpi=300, bbox_inches='tight')
            plt.close()

print("\n완료!")

In [ ]:
def get_cross_track(pixc_file):
    ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    ct = ds['cross_track'].values
    ds.close()
    return ct.min(), ct.max()

spikes = d_sorted[d_sorted['is_spike']].copy()

for idx, spike_row in spikes.iterrows():
    previous_normals = d_sorted[(d_sorted['time_dt'] < spike_row['time_dt']) & (~d_sorted['is_spike'])]
    if len(previous_normals) == 0:
        continue
    
    prev_normal = previous_normals.iloc[-1]
    
    normal_files = match_pixc_files(prev_normal['time_str'], pixc_files, prev_normal['crid'])
    for nf, tile in normal_files:
        ct_min, ct_max = get_cross_track(nf)
        print(f"N {str(prev_normal['time_str'])[:10]} {tile}: {ct_min:.0f}~{ct_max:.0f}")
    
    spike_files = match_pixc_files(spike_row['time_str'], pixc_files, spike_row['crid'])
    for sf, tile in spike_files:
        ct_min, ct_max = get_cross_track(sf)
        print(f"S {str(spike_row['time_str'])[:10]} {tile}: {ct_min:.0f}~{ct_max:.0f}")

In [ ]:
selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
mask_frac = np.isclose(selected_12["fctimp_mea"], 0.45, atol=0.01)
sample_1 = selected_12.loc[mask_frac].copy()


bounds = sample_1.total_bounds  # [minx, miny, maxx, maxy]
bbox = (
    bounds[0] - 0.1,  # west
    bounds[1] - 0.1,  # south
    bounds[2] + 0.1,  # east
    bounds[3] + 0.1   # north
)

print(f"bounding_box={bbox}")

fctimp_001 = pd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/1_data_processing/fctimp_mean_0.45_sample.parquet')
unique_dates = fctimp_001['time_str'].unique()  # 'YYYY-MM-DD'
dates = set(pd.to_datetime(unique_dates).date)

earthaccess.login() 

# earthaccess.login() 

# results = earthaccess.search_data(
#     short_name='SWOT_L2_HR_PIXC_2.0',  # Version D
#     temporal=('2024-01-01', '2024-01-31'),
#     bounding_box=(-74, 40, -73, 41)  # (west, south, east, north)
# )

all_results = []
for date in dates:
    date_str = date.strftime('%Y-%m-%d')
    results = earthaccess.search_data(
        short_name='SWOT_L2_HR_PIXC_2.0',
        bounding_box=bbox,
        temporal=(date_str, date_str)
    )
    all_results.extend(results)
    # print(f"{date_str}: {len(results)} granules")

print(f"\nTotal: {len(all_results)} granules")

with open('swot-pixel_urls_045.txt', 'w') as f:
    for r in all_results:
        f.write(r.data_links()[0] + '\n')

In [ ]:
d = fctimp_001.copy()

swot_col = "wse_relative_m"
gage_col = "wse_usgs_relative_m"
err_name = "error_relative_m"

d[err_name] = (d[swot_col] - d[gage_col]).abs()
spike_mask = d[err_name] >= 1
d_spike = d.loc[spike_mask].copy()

d_spike

### error analysis

In [ ]:
import earthaccess
import pandas as pd
import numpy as np
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta
from shapely.vectorized import contains
from shapely import contains_xy

# -------------------------
# Small debug helpers
# -------------------------
def dbg(msg):
    print(f"[DEBUG] {msg}")

def assert_cols(df, cols, name="DataFrame"):
    missing = [c for c in cols if c not in df.columns]
    assert len(missing) == 0, f"{name} missing columns: {missing}"

def safe_get_var(ds, candidates, var_label):
    """
    Try multiple variable names in ds.
    """
    for c in candidates:
        if c in ds.variables:
            return ds[c].values
    raise KeyError(f"{var_label} not found in dataset. Tried {candidates}")


In [ ]:
selected_12 = gpd.read_file('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_12_vD.shp')
assert len(selected_12) > 0, "selected_12 shapefile is empty."
assert_cols(selected_12, ["fctimp_mea"], name="selected_12")

mask_frac = np.isclose(selected_12["fctimp_mea"], 0.00607272, atol=0.01)
sample_1 = selected_12.loc[mask_frac].copy()

assert len(sample_1) > 0, "No reaches matched fctimp_mea filter; check atol or target value."
assert sample_1.geometry.notnull().all(), "sample_1 has null geometries."

dbg(f"sample_1 size = {len(sample_1)}")

In [ ]:
bounds = sample_1.total_bounds  # [minx, miny, maxx, maxy]
bbox = (bounds[0]-0.1, bounds[1]-0.1, bounds[2]+0.1, bounds[3]+0.1)

dbg(f"bounding_box={bbox}")

In [ ]:
fctimp_001 = pd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/1_data_processing/fctimp_mean_0.01_sample.parquet')
assert len(fctimp_001) > 0, "fctimp_001 parquet is empty."

required_cols = [
    "time_str", "time_dt", "crid",
    "wse_relative_m", "wse_usgs_relative_m",
    "fctimp_mean"
]
assert_cols(fctimp_001, required_cols, name="fctimp_001")

d = fctimp_001.copy()
d["error"] = (d["wse_relative_m"] - d["wse_usgs_relative_m"]).abs()
d["is_spike"] = d["error"] >= 1.0

dbg(f"Total rows={len(d)} | spikes={d['is_spike'].sum()} | normals={(~d['is_spike']).sum()}")

assert d["time_dt"].notnull().all(), "time_dt has nulls; cannot sort by time."
d_sorted = d.sort_values("time_dt").reset_index(drop=True)

In [ ]:
pixc_dir = Path('/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_pixel_001')
assert pixc_dir.exists(), f"pixc_dir not found: {pixc_dir}"

pixc_files = sorted(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc'))
assert len(pixc_files) > 0, "No PIXC files found in pixc_dir."

dbg(f"Found {len(pixc_files)} PIXC files.")

In [ ]:
def match_pixc_files(time_str, files, crid, window_minutes=5, allowed_tiles=None):
    dt = pd.to_datetime(time_str)
    if dt.tzinfo is not None:
        dt = dt.tz_localize(None)
    start_time = dt - timedelta(minutes=window_minutes)
    end_time = dt + timedelta(minutes=window_minutes)
    crid = str(crid)
    
    matched = []
    for f in files:
        parts = f.name.split('_')
        if len(parts) < 10:
            continue
        try:
            file_start = pd.to_datetime(parts[7], format='%Y%m%dT%H%M%S')
            file_end = pd.to_datetime(parts[8], format='%Y%m%dT%H%M%S')
            file_crid = parts[9]
            tile_info = parts[6]
            
            if allowed_tiles is not None and tile_info not in allowed_tiles:
                continue
            time_overlap = not (file_end < start_time or file_start > end_time)
            if time_overlap and (file_crid == crid):
                matched.append((f, tile_info))
        except Exception:
            continue
    
    from collections import defaultdict
    by_tile = defaultdict(list)
    for f, tile in matched:
        by_tile[tile].append(f)
    
    result = []
    for tile, flist in by_tile.items():
        latest = sorted(flist, key=lambda x: x.name)[-1]
        result.append((latest, tile))
    
    return result

In [ ]:
polygon = sample_1.to_crs('EPSG:4326')
poly_geom = polygon.geometry.iloc[0]
poly_bounds = polygon.total_bounds  # lon/lat bounds

def extract_pixels(pixc_file, polygon_gdf):
    ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    try:
        lon = ds['longitude'].values
        lat = ds['latitude'].values
        assert lon.ndim == 1 and lat.ndim == 1
        
        bounds = polygon_gdf.total_bounds
        bbox_mask = ((lon >= bounds[0]) & (lon <= bounds[2]) &
                     (lat >= bounds[1]) & (lat <= bounds[3]))
        if bbox_mask.sum() == 0:
            return None
        
        classification = ds['classification'].values[bbox_mask]
        water_mask = classification > 2
        if water_mask.sum() == 0:
            return None
        
        lon_filt = lon[bbox_mask][water_mask]
        lat_filt = lat[bbox_mask][water_mask]
        poly = polygon_gdf.geometry.iloc[0]
        inside_mask = contains_xy(poly, lon_filt, lat_filt)
        if inside_mask.sum() == 0:
            return None
        
        final_idx = np.where(bbox_mask)[0][water_mask][inside_mask]
        
        # Extract all variables
        result = {
            "lon": lon[final_idx],
            "lat": lat[final_idx],
            "lon_all": lon,
            "lat_all": lat,
            "final_idx": final_idx,
            
            # Core measurements
            "height": safe_get_var(ds, ["height"], "height")[final_idx],
            "classification": ds["classification"].values[final_idx],
            
            # Quality flags (uint32)
            "interferogram_qual": safe_get_var(ds, ["interferogram_qual"], "interferogram_qual")[final_idx],
            "classification_qual": safe_get_var(ds, ["classification_qual"], "classification_qual")[final_idx],
            "geolocation_qual": safe_get_var(ds, ["geolocation_qual"], "geolocation_qual")[final_idx],
            "sig0_qual": safe_get_var(ds, ["sig0_qual"], "sig0_qual")[final_idx],
            
            # Power/channel
            "power_plus_y": safe_get_var(ds, ["power_plus_y"], "power_plus_y")[final_idx],
            "power_minus_y": safe_get_var(ds, ["power_minus_y"], "power_minus_y")[final_idx],
            "x_factor_plus_y": safe_get_var(ds, ["x_factor_plus_y"], "x_factor_plus_y")[final_idx],
            "x_factor_minus_y": safe_get_var(ds, ["x_factor_minus_y"], "x_factor_minus_y")[final_idx],
            "coh_power": safe_get_var(ds, ["coherent_power"], "coherent_power")[final_idx],
            "sig0": safe_get_var(ds, ["sig0"], "sig0")[final_idx],
            
            # Water detection
            "bright_land": safe_get_var(ds, ["bright_land_flag"], "bright_land_flag")[final_idx],
            "water_frac": safe_get_var(ds, ["water_frac"], "water_frac")[final_idx],
            "false_detection_rate": safe_get_var(ds, ["false_detection_rate"], "false_detection_rate")[final_idx],
            "missed_detection_rate": safe_get_var(ds, ["missed_detection_rate"], "missed_detection_rate")[final_idx],
            "prior_water_prob": safe_get_var(ds, ["prior_water_prob"], "prior_water_prob")[final_idx],
            # "prior_water_change": safe_get_var(ds, ["prior_water_change"], "prior_water_change")[final_idx],
            
            # Geometry
            "layover": safe_get_var(ds, ["layover_impact"], "layover_impact")[final_idx],
            "cross_track": safe_get_var(ds, ["cross_track"], "cross_track")[final_idx],
            "pixel_area": safe_get_var(ds, ["pixel_area"], "pixel_area")[final_idx],
            "inc": safe_get_var(ds, ["inc"], "inc")[final_idx],
            "phase_noise_std": safe_get_var(ds, ["phase_noise_std"], "phase_noise_std")[final_idx],
            
            # Looks
            "eff_num_rare_looks": safe_get_var(ds, ["eff_num_rare_looks"], "eff_num_rare_looks")[final_idx],
            "eff_num_medium_looks": safe_get_var(ds, ["eff_num_medium_looks"], "eff_num_medium_looks")[final_idx],
            
            # Phase unwrapping
            "phase_unwrapping_region": safe_get_var(ds, ["phase_unwrapping_region"], "phase_unwrapping_region")[final_idx],
            
            # Indices
            "azimuth": safe_get_var(ds, ["azimuth_index"], "azimuth_index")[final_idx],
            "range": safe_get_var(ds, ["range_index"], "range_index")[final_idx],
            
            # Sensitivities
            "dheight_dphase": safe_get_var(ds, ["dheight_dphase"], "dheight_dphase")[final_idx],
            "dheight_droll": safe_get_var(ds, ["dheight_droll"], "dheight_droll")[final_idx],
            "dheight_dbaseline": safe_get_var(ds, ["dheight_dbaseline"], "dheight_dbaseline")[final_idx],
            
            # Corrections
            "model_dry_tropo_cor": safe_get_var(ds, ["model_dry_tropo_cor"], "model_dry_tropo_cor")[final_idx],
            "model_wet_tropo_cor": safe_get_var(ds, ["model_wet_tropo_cor"], "model_wet_tropo_cor")[final_idx],
            "iono_cor_gim_ka": safe_get_var(ds, ["iono_cor_gim_ka"], "iono_cor_gim_ka")[final_idx],
            "height_cor_xover": safe_get_var(ds, ["height_cor_xover"], "height_cor_xover")[final_idx],
            "sig0_cor_atmos_model": safe_get_var(ds, ["sig0_cor_atmos_model"], "sig0_cor_atmos_model")[final_idx],
        }
        
        return result
    finally:
        ds.close()

In [ ]:
def safe_mean(arr):
    """Return mean or 0.0 if empty array"""
    return 0.0 if len(arr) == 0 else float(arr.mean())

def check_bit(flags, bit_position):
    """Check if specific bit is set (0-indexed from LSB)"""
    flags = flags.astype(np.uint32)  
    return (flags & (1 << bit_position)) > 0

def get_exclusion_mask(data):
    """Create mask for system event exclusion (bits 10,13,14,15,18,29,30,31)"""
    exclude_bits = [10, 13, 14, 15, 18, 29, 30, 31]
    mask = np.ones(len(data["classification"]), dtype=bool)
    
    for qual_name in ["interferogram_qual", "classification_qual", "geolocation_qual", "sig0_qual"]:
        if qual_name in data and data[qual_name] is not None:
            qual = data[qual_name].astype(np.uint32)
            for bit in exclude_bits:
                mask &= ~check_bit(qual, bit)
    return mask

def compute_metrics(data, use_exclusion=True):
    n_total = len(data["classification"])
    
    if use_exclusion:
        mask = get_exclusion_mask(data)
    else:
        mask = np.ones(n_total, dtype=bool)
    
    n_valid = int(mask.sum())
    
    def get(key):
        return data[key][mask] if key in data and data[key] is not None else None
    
    metrics = {"n_total": n_total, "n_valid": n_valid}
    
    # 1. BRIGHT LAND
    cq = get("classification_qual")
    if cq is not None:
        metrics["detected_water_but_bright_land"] = safe_mean(check_bit(cq, 3))
    
    bright = get("bright_land")
    if bright is not None:
        for val in [0, 1, 2]:
            metrics[f"bright_land_{val}"] = safe_mean(bright == val)
    
    cls = get("classification")
    if cls is not None:
        metrics["class_7_open_low_coh"] = safe_mean(cls == 7)
    
    for key in ["prior_water_prob", "false_detection_rate", "missed_detection_rate", 
                "water_frac"]:  # 추가
        val = get(key)
        if val is not None:
            metrics[key] = val
    
    # 2. LAYOVER
    gq = get("geolocation_qual")
    if gq is not None:
        metrics["layover_significant"] = safe_mean(check_bit(gq, 0))
    
    lay = get("layover")
    if lay is not None:
        metrics["layover_impact"] = lay
    
    # 3. SPECULAR RINGING
    ringing_mask = np.zeros(n_valid, dtype=bool) if n_valid > 0 else np.array([], dtype=bool)
    for qual_name, bit in [("interferogram_qual", 19), ("classification_qual", 19), 
                            ("geolocation_qual", 19), ("sig0_qual", 19)]:
        q = get(qual_name)
        if q is not None:
            ringing_mask |= check_bit(q, bit)
    metrics["specular_ringing_any"] = safe_mean(ringing_mask)
    
    # 4. COHERENT POWER / SIG0
    iq = get("interferogram_qual")
    if iq is not None:
        metrics["no_coherent_gain"] = safe_mean(check_bit(iq, 0))
        metrics["power_close_to_noise_floor"] = safe_mean(check_bit(iq, 1))
        metrics["coherent_power_suspect"] = safe_mean(check_bit(iq, 11))
        metrics["coherent_power_bad"] = safe_mean(check_bit(iq, 27))
    
    sq = get("sig0_qual")
    if sq is not None:
        metrics["sig0_uncert_suspect"] = safe_mean(check_bit(sq, 0))
        metrics["noise_power_suspect"] = safe_mean(check_bit(sq, 2))
        metrics["xfactor_suspect"] = safe_mean(check_bit(sq, 3))
        metrics["noise_power_bad"] = safe_mean(check_bit(sq, 25))
        metrics["xfactor_bad"] = safe_mean(check_bit(sq, 26))
    
    for key in ["coh_power", "sig0", "phase_noise_std", "sig0_uncert",
                "eff_num_rare_looks", "eff_num_medium_looks", "inc", "pixel_area"]:  # 추가
        val = get(key)
        if val is not None:
            metrics[key] = val
    
    # 5. POWER ASYMMETRY
    ppy = get("power_plus_y")
    pmy = get("power_minus_y")
    if ppy is not None and pmy is not None:
        metrics["power_plus_y"] = ppy
        metrics["power_minus_y"] = pmy
        metrics["power_diff"] = ppy - pmy
    
    xpy = get("x_factor_plus_y")
    xmy = get("x_factor_minus_y")
    if xpy is not None and xmy is not None:
        metrics["x_factor_plus_y"] = xpy
        metrics["x_factor_minus_y"] = xmy
        metrics["x_factor_diff"] = xpy - xmy
    
    # 6. PHASE UNWRAPPING
    if gq is not None:
        metrics["phase_noise_suspect"] = safe_mean(check_bit(gq, 2))
        metrics["phase_unwrapping_suspect"] = safe_mean(check_bit(gq, 4))
        metrics["medium_phase_suspect"] = safe_mean(check_bit(gq, 12))
        metrics["geolocation_is_from_refloc"] = safe_mean(check_bit(gq, 24))
        metrics["no_geolocation_bad"] = safe_mean(check_bit(gq, 26))
        metrics["medium_phase_bad"] = safe_mean(check_bit(gq, 28))
    
    ct = get("cross_track")
    if ct is not None:
        metrics["cross_track"] = ct
    
    # 7. RAIN
    if sq is not None:
        metrics["sig0_cor_atmos_suspect"] = safe_mean(check_bit(sq, 1))
        metrics["sig0_cor_atmos_missing"] = safe_mean(check_bit(sq, 20))
    
    sc = get("sig0_cor_atmos_model")
    if sc is not None:
        metrics["sig0_cor_atmos_model"] = sc
    
    return metrics

In [ ]:
# 1. 전체 매칭 확인
def verify_matching(d_sorted, pixc_files, allowed_tiles={'222R', '087L'}):
    results = []
    
    for idx, row in d_sorted.iterrows():
        matched = match_pixc_files(
            row['time_str'], 
            pixc_files, 
            row['crid'],
            allowed_tiles=allowed_tiles
        )
        
        for pf, tile in matched:
            results.append({
                'idx': idx,
                'time_str': row['time_str'],
                'crid': row['crid'],
                'is_spike': row['is_spike'],
                'error': row['error'],
                'tile': tile,
                'filename': pf.name,
                'label': f"SPIKE_{tile}" if row['is_spike'] else f"NORMAL_{tile}"
            })
    
    return pd.DataFrame(results)

verify_df = verify_matching(d_sorted, pixc_files)

# 2. 체크
print(f"d_sorted rows: {len(d_sorted)}")
print(f"matched files: {len(verify_df)}")
print(f"\nBy label:")
print(verify_df['label'].value_counts())

# 3. Product counter 체크 (같은 time/crid에 중복 있는지)
duplicates = verify_df.groupby(['time_str', 'crid', 'tile']).size()
duplicates = duplicates[duplicates > 1]
if len(duplicates) > 0:
    print(f"\n⚠️  Duplicates found: {len(duplicates)}")
    print(duplicates)
else:
    print("\n✓ No duplicates (product counter filter working)")

# 4. SPIKE/NORMAL 분류 체크
print(f"\nSPIKE classification:")
print(f"  d_sorted spikes: {d_sorted['is_spike'].sum()}")
print(f"  verify_df spikes: {verify_df['is_spike'].sum()}")

verify_df

In [ ]:
def collect_all_samples(d_sorted, pixc_files, sample_1):
    samples = defaultdict(list)
    
    for _, row in d_sorted.iterrows():
        matched = match_pixc_files(row['time_str'], pixc_files, row['crid'], 
                                   allowed_tiles={'222R', '087L'})
        for pf, tile in matched:
            data = extract_pixels(pf, sample_1)
            if data:
                label = f"SPIKE_{tile}" if row['is_spike'] else f"NORMAL_{tile}"
                m_unfilt = compute_metrics(data, False)
                m_filt = compute_metrics(data, True)
                m_unfilt['error'] = np.array([row['error']])
                m_filt['error'] = np.array([row['error']])
                samples[label].append(m_unfilt)
                samples[label + '_filt'].append(m_filt)
    
    return samples

samples_all = collect_all_samples(d_sorted, pixc_files, sample_1)

In [ ]:
for key in ['SPIKE_222R', 'NORMAL_222R', 'NORMAL_087L']:
    print(f"{key}: {len(samples_all[key])} samples")

In [ ]:
def plot_overview(samples_all):
    binary_keys = [
        'detected_water_but_bright_land', 'bright_land_0', 'bright_land_1', 'bright_land_2',
        'class_7_open_low_coh', 'layover_significant', 'specular_ringing_any',
        'no_coherent_gain', 'power_close_to_noise_floor', 'coherent_power_suspect',
        'coherent_power_bad', 'sig0_uncert_suspect', 'noise_power_suspect',
        'xfactor_suspect', 'phase_unwrapping_suspect', 'sig0_cor_atmos_suspect'
    ]
    
    continuous_keys = [
        'error', 'layover_impact', 'coh_power', 'sig0', 'phase_noise_std',
        'power_plus_y', 'power_minus_y', 'power_diff', 
        'x_factor_plus_y', 'x_factor_minus_y', 'x_factor_diff',
        'cross_track', 'prior_water_prob', 'false_detection_rate'
    ]
    
    classes = ['SPIKE_222R', 'NORMAL_222R', 'NORMAL_087L']
    colors = ['red', 'blue', 'green']
    
    # === BAR GRAPHS (separate files) ===
    for use_filt in [False, True]:
        suffix = '_filt' if use_filt else ''
        title = 'Filtered' if use_filt else 'Unfiltered'
        
        fig, ax = plt.subplots(1, 1, figsize=(16, 6))
        
        data_matrix = []
        for cls in classes:
            row = []
            for key in binary_keys:
                vals = [m.get(key, 0) for m in samples_all[cls + suffix] if key in m]
                row.append(np.mean(vals) if vals else 0)
            data_matrix.append(row)
        
        x = np.arange(len(binary_keys))
        width = 0.25
        for i, (cls, color) in enumerate(zip(classes, colors)):
            ax.bar(x + i*width, data_matrix[i], width, label=cls, color=color, alpha=0.7)
        
        ax.set_ylabel('Rate', fontsize=11)
        ax.set_title(f'Binary/Categorical Flags - {title}', fontsize=13, fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(binary_keys, rotation=45, ha='right', fontsize=9)
        ax.legend(fontsize=10)
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim(0, 1.1)
        
        plt.tight_layout()
        plt.savefig(f'binary_flags{suffix}.png', dpi=150, bbox_inches='tight')
        plt.show()
    
    # === BOXPLOTS ===
    n_vars = len(continuous_keys)
    n_cols = 4
    n_rows = (n_vars + n_cols - 1) // n_cols
    
    for use_filt in [False, True]:
        suffix = '_filt' if use_filt else ''
        title = 'Filtered' if use_filt else 'Unfiltered'
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4*n_rows))
        axes = axes.flatten()
        
        for idx, key in enumerate(continuous_keys):
            ax = axes[idx]
            box_data = []
            
            for cls in classes:
                vals = []
                for m in samples_all[cls + suffix]:
                    if key in m and m[key] is not None:
                        arr = np.asarray(m[key])
                        if arr.size > 0:
                            vals.extend(arr.flatten()[~np.isnan(arr.flatten())])
                box_data.append(vals)
            
            if any(len(d) > 0 for d in box_data):
                bp = ax.boxplot(box_data, tick_labels=classes, patch_artist=True, showfliers=False)
                for patch, color in zip(bp['boxes'], colors):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.6)
            
            ax.set_title(key, fontsize=10, fontweight='bold')
            ax.tick_params(axis='x', rotation=15, labelsize=8)
            ax.grid(axis='y', alpha=0.3)
        
        for idx in range(len(continuous_keys), len(axes)):
            axes[idx].axis('off')
        
        fig.suptitle(f'Continuous Variables - {title}', fontsize=15, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'continuous_boxplots{suffix}.png', dpi=150, bbox_inches='tight')
        plt.show()

plot_overview(samples_all)

In [ ]:
def plot_222R_comparison(samples_all):
    binary_keys = [
        'detected_water_but_bright_land', 'bright_land_0', 'bright_land_1', 'bright_land_2',
        'class_7_open_low_coh', 'layover_significant', 'specular_ringing_any',
        'no_coherent_gain', 'power_close_to_noise_floor', 'coherent_power_suspect',
        'coherent_power_bad', 'sig0_uncert_suspect', 'noise_power_suspect',
        'xfactor_suspect', 'phase_unwrapping_suspect', 'sig0_cor_atmos_suspect'
    ]
    
    continuous_keys = [
        'error', 'layover_impact', 'coh_power', 'sig0', 'phase_noise_std',
        'power_plus_y', 'power_minus_y', 'power_diff', 
        'x_factor_plus_y', 'x_factor_minus_y', 'x_factor_diff',
        'cross_track', 'prior_water_prob', 'false_detection_rate','inc','missed_detection_rate'
    ]
    
    classes = ['SPIKE_222R', 'NORMAL_222R']
    colors = ['red', 'blue']
    
    # BAR GRAPHS
    for use_filt in [False, True]:
        suffix = '_filt' if use_filt else ''
        title = 'Filtered' if use_filt else 'Unfiltered'
        
        fig, ax = plt.subplots(1, 1, figsize=(16, 6))
        
        data_matrix = []
        for cls in classes:
            row = []
            for key in binary_keys:
                vals = [m.get(key, 0) for m in samples_all[cls + suffix] if key in m]
                row.append(np.mean(vals) if vals else 0)
            data_matrix.append(row)
        
        x = np.arange(len(binary_keys))
        width = 0.35
        for i, (cls, color) in enumerate(zip(classes, colors)):
            ax.bar(x + i*width, data_matrix[i], width, label=cls, color=color, alpha=0.7)
        
        ax.set_ylabel('Rate', fontsize=11)
        ax.set_title(f'222R Only: Binary Flags - {title}', fontsize=13, fontweight='bold')
        ax.set_xticks(x + width/2)
        ax.set_xticklabels(binary_keys, rotation=45, ha='right', fontsize=9)
        ax.legend(fontsize=11)
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim(0, 1.1)
        
        plt.tight_layout()
        plt.savefig(f'222R_binary{suffix}.png', dpi=150, bbox_inches='tight')
        plt.show()
    
    # BOXPLOTS
    n_vars = len(continuous_keys)
    n_cols = 4
    n_rows = (n_vars + n_cols - 1) // n_cols
    
    for use_filt in [False, True]:
        suffix = '_filt' if use_filt else ''
        title = 'Filtered' if use_filt else 'Unfiltered'
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
        axes = axes.flatten()
        
        for idx, key in enumerate(continuous_keys):
            ax = axes[idx]
            box_data = []
            
            for cls in classes:
                vals = []
                for m in samples_all[cls + suffix]:
                    if key in m and m[key] is not None:
                        arr = np.asarray(m[key])
                        if arr.size > 0:
                            vals.extend(arr.flatten()[~np.isnan(arr.flatten())])
                box_data.append(vals)
            
            if any(len(d) > 0 for d in box_data):
                bp = ax.boxplot(box_data, tick_labels=classes, patch_artist=True, showfliers=False)
                for patch, color in zip(bp['boxes'], colors):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.6)
            
            ax.set_title(key, fontsize=10, fontweight='bold')
            ax.tick_params(axis='x', rotation=15, labelsize=8)
            ax.grid(axis='y', alpha=0.3)
        
        for idx in range(len(continuous_keys), len(axes)):
            axes[idx].axis('off')
        
        fig.suptitle(f'222R Only: Continuous Variables - {title}', fontsize=15, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'222R_continuous{suffix}.png', dpi=150, bbox_inches='tight')
        plt.show()

plot_222R_comparison(samples_all)

In [ ]:
def plot_222R_samples(samples_all):
    # 모든 222R 샘플 수집 & error 기준 정렬
    all_samples = []
    for label in ['SPIKE_222R', 'NORMAL_222R']:
        for m in samples_all[label]:
            error_val = m['error'][0] if 'error' in m and len(m['error']) > 0 else 0
            all_samples.append((error_val, m, label))
    
    all_samples.sort(key=lambda x: x[0], reverse=True)
    
    binary_keys = ['class_7_open_low_coh', 'layover_significant', 'specular_ringing_any']
    continuous_keys = ['error', 'layover_impact', 'coh_power', 'sig0', 'phase_noise_std',
                       'power_plus_y', 'power_minus_y', 'power_diff', 
                       'x_factor_plus_y', 'x_factor_minus_y', 'x_factor_diff',
                       'false_detection_rate', 'inc', 'inc','missed_detection_rate']
    
    n_samples = len(all_samples)
    n_vars = len(binary_keys) + len(continuous_keys)
    
    fig, axes = plt.subplots(n_vars, 1, figsize=(16, 2*n_vars))
    
    x_labels = [f"{err:.2f}" for err, _, _ in all_samples]
    x_pos = np.arange(n_samples)
    colors = ['red' if label == 'SPIKE_222R' else 'blue' for _, _, label in all_samples]
    
    row_idx = 0
    
    # Binary flags
    for key in binary_keys:
        ax = axes[row_idx]
        vals = [m.get(key, 0) for _, m, _ in all_samples]
        ax.bar(x_pos, vals, color=colors, alpha=0.7, width=0.8)
        ax.set_ylabel(key, fontsize=9)
        ax.set_ylim(0, 1.1)
        ax.grid(axis='y', alpha=0.3)
        if row_idx == 0:
            ax.set_title('222R Samples Sorted by Error (Red=SPIKE, Blue=NORMAL)', fontweight='bold')
        row_idx += 1
    
    # Continuous
    for key in continuous_keys:
        ax = axes[row_idx]
        box_data = []
        for _, m, _ in all_samples:
            if key in m and m[key] is not None:
                arr = np.asarray(m[key]).flatten()
                arr = arr[~np.isnan(arr)]
                box_data.append(arr if len(arr) > 0 else [0])
            else:
                box_data.append([0])
        
        bp = ax.boxplot(box_data, positions=x_pos, widths=0.6, 
                        patch_artist=True, showfliers=False)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        
        ax.set_ylabel(key, fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        row_idx += 1
    
    axes[-1].set_xticks(x_pos)
    axes[-1].set_xticklabels(x_labels, rotation=45, ha='right', fontsize=8)
    axes[-1].set_xlabel('Error (m)', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('222R_samples_sorted.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_222R_samples(samples_all)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import timedelta
from shapely.vectorized import contains

d_sorted = d.sort_values('time_dt').reset_index(drop=True)
d_sorted['error'] = (d_sorted['wse_relative_m'] - d_sorted['wse_usgs_relative_m']).abs()
d_sorted['is_spike'] = d_sorted['error'] >= 1.0

pixc_dir = Path('/project/pi_kandread_umass_edu/swot-urban/0_data_collection/SWOT_pixel_001')
pixc_files = list(pixc_dir.glob('SWOT_L2_HR_PIXC_*.nc'))
polygon = sample_1.to_crs('EPSG:4326')
bounds = polygon.total_bounds

def match_pixc_files(time_str, files, crid, window_minutes=5):
    dt = pd.to_datetime(time_str)
    if dt.tzinfo is not None:
        dt = dt.tz_localize(None)
    start_time = dt - timedelta(minutes=window_minutes)
    end_time = dt + timedelta(minutes=window_minutes)
    
    matched = []
    for f in files:
        parts = f.name.split('_')
        if len(parts) >= 10:
            try:
                file_start = pd.to_datetime(parts[7], format='%Y%m%dT%H%M%S')
                file_end = pd.to_datetime(parts[8], format='%Y%m%dT%H%M%S')
                file_crid = parts[9]
                tile_info = parts[6]
                if (not (file_end < start_time or file_start > end_time)) and (file_crid == crid):
                    matched.append((f, tile_info))
            except:
                continue
    return matched

def extract_pixels(pixc_file, polygon_gdf):
    ds = xr.open_dataset(pixc_file, group='pixel_cloud')
    lon = ds['longitude'].values
    lat = ds['latitude'].values
    bounds = polygon_gdf.total_bounds
    
    bbox_mask = ((lon >= bounds[0]) & (lon <= bounds[2]) & 
                 (lat >= bounds[1]) & (lat <= bounds[3]))
    if bbox_mask.sum() == 0:
        ds.close()
        return None
    
    classification = ds['classification'].values[bbox_mask]
    water_mask = classification > 2
    if water_mask.sum() == 0:
        ds.close()
        return None
    
    lon_filt = lon[bbox_mask][water_mask]
    lat_filt = lat[bbox_mask][water_mask]
    poly = polygon_gdf.geometry.iloc[0]
    inside_mask = contains(poly, lon_filt, lat_filt)
    if inside_mask.sum() == 0:
        ds.close()
        return None
    
    final_idx = np.where(bbox_mask)[0][water_mask][inside_mask]
    result = {
        'lon': lon[final_idx],
        'lat': lat[final_idx],
        'lon_all': lon,
        'lat_all': lat,
        'height': ds['height'].values[final_idx],
        'classification': ds['classification'].values[final_idx],
        'bright_land': ds['bright_land_flag'].values[final_idx],
        'layover': ds['layover_impact'].values[final_idx],
        'sig0': ds['sig0'].values[final_idx],
        'coh_power': ds['coherent_power'].values[final_idx],
        'azimuth': ds['azimuth_index'].values[final_idx],
        'range': ds['range_index'].values[final_idx]
    }
    ds.close()
    return result

def plot_height(data, poly, label):
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(data['lon'], data['lat'], c=data['height'], 
                    cmap='viridis', s=0.01, alpha=0.7)
    poly.boundary.plot(ax=ax, color='red', linewidth=2)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'{label} - Height')
    plt.colorbar(sc, ax=ax, label='Height (m)')
    plt.tight_layout()
    return fig

def plot_classification(data, poly, label):
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ['#8B4513', '#D2B48C', '#32CD32', '#FF0000', '#00008B', '#808080', '#87CEEB']
    labels = ['land', 'land_near_water', 'water_near_land', 'open_water', 
              'dark_water', 'low_coh_water_near_land', 'open_low_coh_water']
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch
    
    sc = ax.scatter(data['lon'], data['lat'], c=data['classification'], 
                    cmap=ListedColormap(colors), s=0.01, alpha=0.8, vmin=1, vmax=7)
    poly.boundary.plot(ax=ax, color='black', linewidth=2)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'{label} - Classification')
    
    legend = [Patch(facecolor=colors[i], label=labels[i]) for i in range(7)]
    ax.legend(handles=legend, loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)
    plt.tight_layout()
    return fig

def plot_tile_overview(data, poly, label):
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(data['lon_all'], data['lat_all'], c='lightgray', s=0.1, alpha=0.3)
    ax.scatter(data['lon'], data['lat'], c='blue', s=0.01, alpha=0.5)
    poly.boundary.plot(ax=ax, color='red', linewidth=3)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'{label} - Tile Overview with Polygon')
    plt.tight_layout()
    return fig

def plot_statistics(data, label):
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    # 1. Classification count
    class_counts = [np.sum(data['classification'] == i) for i in range(1, 8)]
    axes[0].bar(range(1, 8), class_counts, color='steelblue')
    axes[0].set_xlabel('Classification')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Classification Count')
    axes[0].grid(alpha=0.3)
    
    # 2. Bright land flag count
    bright_counts = [np.sum(data['bright_land'] == i) for i in range(2)]
    axes[1].bar(range(2), bright_counts, color='orange')
    axes[1].set_xlabel('Bright Land Flag')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Bright Land Flag Count')
    axes[1].set_xticks([0, 1])
    axes[1].grid(alpha=0.3)
    
    # 3. Layover impact boxplot
    axes[2].boxplot(data['layover'], vert=True)
    axes[2].set_ylabel('Layover Impact')
    axes[2].set_title('Layover Impact')
    axes[2].grid(alpha=0.3)
    
    # 4. Sigma0 boxplot
    axes[3].boxplot(data['sig0'], vert=True)
    axes[3].set_ylabel('Sigma0 (dB)')
    axes[3].set_title('Sigma0')
    axes[3].grid(alpha=0.3)
    
    # 5. Coherent power boxplot
    axes[4].boxplot(data['coh_power'], vert=True)
    axes[4].set_ylabel('Coherent Power')
    axes[4].set_title('Coherent Power')
    axes[4].grid(alpha=0.3)
    
    # 6. Total pixel count
    axes[5].bar([0], [len(data['lon'])], color='green')
    axes[5].set_ylabel('Count')
    axes[5].set_title(f'Total Pixels: {len(data["lon"])}')
    axes[5].set_xticks([])
    axes[5].grid(alpha=0.3)
    
    # 7. Azimuth boxplot
    axes[6].boxplot(data['azimuth'], vert=True)
    axes[6].set_ylabel('Azimuth Index')
    axes[6].set_title('Azimuth Index')
    axes[6].grid(alpha=0.3)
    
    # 8. Range boxplot
    axes[7].boxplot(data['range'], vert=True)
    axes[7].set_ylabel('Range Index')
    axes[7].set_title('Range Index')
    axes[7].grid(alpha=0.3)
    
    fig.suptitle(label, fontsize=14, y=1.00)
    plt.tight_layout()
    return fig

In [ ]:
# error 5.47과 0.88인 샘플 찾기
target_errors = [5.47, 0.88]
selected_rows = []

for target in target_errors:
    row = d_sorted.iloc[(d_sorted['error'] - target).abs().argsort()[0]]
    selected_rows.append(row)
    print(f"Target {target}m → Found error={row['error']:.2f}m, time={row['time_str']}, crid={row['crid']}")

# 각 샘플 그리기
for row in selected_rows:
    label_prefix = 'SPIKE' if row['is_spike'] else 'NORMAL'
    matched = match_pixc_files(row['time_str'], pixc_files, row['crid'])
    
    for pf, tile in matched:
        data = extract_pixels(pf, polygon)
        if data:
            time_str = row['time_dt'].strftime('%Y%m%dT%H%M%S')
            crid = row['crid']
            
            # Height
            fig = plot_height(data, polygon, f'{label_prefix}-{time_str}-{crid}-{tile}')
            plt.savefig(f'{label_prefix}_{time_str}_{crid}_{tile}_Height.png', dpi=300, bbox_inches='tight')
            plt.close()
            
            # Classification
            fig = plot_classification(data, polygon, f'{label_prefix}-{time_str}-{crid}-{tile}')
            plt.savefig(f'{label_prefix}_{time_str}_{crid}_{tile}_Classification.png', dpi=300, bbox_inches='tight')
            plt.close()
            
            # Overview
            fig = plot_tile_overview(data, polygon, f'{label_prefix}-{time_str}-{crid}-{tile}')
            plt.savefig(f'{label_prefix}_{time_str}_{crid}_{tile}_Overview.png', dpi=300, bbox_inches='tight')
            plt.close()

            fig = plot_statistics(data, f'{label_prefix}-{time_str}-{crid}-{tile}')
            plt.savefig(f'{label_prefix}_{time_str}_{crid}_{tile}_Statistics.png', dpi=300, bbox_inches='tight')
            plt.close()
            
            print(f"Saved: {label_prefix}_{time_str}_{crid}_{tile}")